Environment Check.

In [ ]:
# Quick SageMaker preflight — checks packages, AWS identity, S3 access, base path, HF streaming
import os
import importlib

print("=== OpenFake SageMaker Preflight Check ===\n")

# 1) Package checks
required = ["boto3", "tqdm", "datasets"]
missing = []

for pkg in required:
    try:
        importlib.import_module(pkg)
        print(f"[OK] Package available: {pkg}")
    except Exception as e:
        print(f"[MISSING] {pkg} -> {e}")
        missing.append(pkg)

# 2) Base path check
base_dir = os.environ.get("OPENFAKE_BASE_DIR", "/home/ec2-user/SageMaker")
print(f"\n[INFO] OPENFAKE_BASE_DIR = {base_dir}")
print(f"[INFO] Base path exists: {os.path.exists(base_dir)}")

# 3) AWS identity + S3 bucket check
aws_ok = True
bucket_name = "deepfake-d-100k-dataset-tw26"

try:
    import boto3

    sts = boto3.client("sts")
    ident = sts.get_caller_identity()
    print(f"\n[OK] AWS identity detected")
    print(f"     Account: {ident.get('Account')}")
    print(f"     ARN: {ident.get('Arn')}")

    s3 = boto3.client("s3")
    s3.head_bucket(Bucket=bucket_name)
    print(f"[OK] S3 bucket accessible: {bucket_name}")

except Exception as e:
    aws_ok = False
    print(f"\n[FAIL] AWS/S3 check failed: {e}")

# 4) Hugging Face streaming probe
hf_ok = True
try:
    from datasets import load_dataset

    ds = load_dataset("ComplexDataLab/OpenFake", split="train", streaming=True)
    first = next(iter(ds))
    print(f"\n[OK] Hugging Face streaming works")
    print(f"     Sample keys: {list(first.keys())}")

except Exception as e:
    hf_ok = False
    print(f"\n[FAIL] Hugging Face streaming failed: {e}")

# 5) Summary
print("\n=== Summary ===")
if missing:
    print(f"[ACTION] Install missing packages: {missing}")
else:
    print("[OK] No missing core packages")

if not os.path.exists(base_dir):
    print("[ACTION] Base directory does not exist. Set OPENFAKE_BASE_DIR to a valid writable path.")

if not aws_ok:
    print("[ACTION] Fix AWS credentials / IAM role / S3 bucket permissions before running downloader.")

if not hf_ok:
    print("[ACTION] Fix Hugging Face connectivity or package issues before running downloader.")

if (not missing) and os.path.exists(base_dir) and aws_ok and hf_ok:
    print("\nGREEN: Safe to run the downloader.")
else:
    print("\nNOT READY: Resolve the issues above first.")

Datasets Installer.

In [1]:
%pip install datasets

  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.8/526.8 kB 29.1 MB/s  0:00:00
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 618.0/618.0 kB 38.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 109.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [datasets]5/6 [datasets]ce-hub]
Note: you may need to restart the kernel to use updated packages.


Download Script for OpenFake - 100K RAW.

HF TOKEN Setup.

In [8]:
# OpenFake AWS Production Downloader.

# Architecture:
#   - Single Hugging Face streaming session downloads both classes in one pass
#   - Real and fake saved separately to local SSD
#   - After download: real folder zipped → uploaded → verified → deleted
#                     fake folder zipped → uploaded → verified → deleted
#   - Separate manifests per class uploaded to S3
#   - base_dir deleted only after both uploads are fully verified

import os
os.environ['HF_DATASETS_DISABLE_PROGRESS_BARS'] = '1'

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("HF_TOKEN is not set in this kernel session.")

from io import BytesIO
from PIL import Image as PILImage, UnidentifiedImageError
import shutil
import random
import json
import boto3
from datetime import datetime, timezone
from datasets import load_dataset
from tqdm.auto import tqdm

random.seed(42)

# CONFIGURATION  —  change these as needed, everything else is derived from them.

TARGET_PER_CLASS   = 50000          # 50K real + 50K fake = 100K raw total
MAX_ITERATIONS     = 600000         # safety ceiling for the HF stream loop
CHECKPOINT_EVERY   = 1000           # write progress checkpoint every N images saved
DISK_MARGIN_FACTOR = 2.2            # require 2.2× the folder size free before zipping

BASE_DIR  = os.environ.get('OPENFAKE_BASE_DIR', '/home/ec2-user/SageMaker')

S3_BUCKET = 'deepfake-d-100k-dataset-tw26'
S3_PREFIX = 'datasets/OpenFake'

# DERIVED PATHS  —  do not hardcode elsewhere

TEMP_RAW_DIR   = os.path.join(BASE_DIR, 'temp_raw')
REAL_DIR       = os.path.join(TEMP_RAW_DIR, 'real')
FAKE_DIR       = os.path.join(TEMP_RAW_DIR, 'fake')
CHECKPOINT_FILE = os.path.join(BASE_DIR, 'openfake_checkpoint.json')

S3_KEYS = {
    'real_zip'       : f'{S3_PREFIX}/openfake_real_raw.zip',
    'fake_zip'       : f'{S3_PREFIX}/openfake_fake_raw.zip',
    'real_manifest'  : f'{S3_PREFIX}/openfake_real_manifest.txt',
    'fake_manifest'  : f'{S3_PREFIX}/openfake_fake_manifest.txt',
}

LOCAL_ZIPS = {
    'real' : os.path.join(BASE_DIR, 'openfake_real_raw'),   # .zip appended by make_archive
    'fake' : os.path.join(BASE_DIR, 'openfake_fake_raw'),
}

MANIFEST_PATHS = {
    'real' : os.path.join(BASE_DIR, 'openfake_real_manifest.txt'),
    'fake' : os.path.join(BASE_DIR, 'openfake_fake_manifest.txt'),
}

# S3 CLIENT

s3 = boto3.client('s3')

# HELPER — CHECKPOINT

def write_checkpoint(iteration, real_count, fake_count, save_errors):
    data = {
        'timestamp'   : datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC'),
        'iteration'   : iteration,
        'real_count'  : real_count,
        'fake_count'  : fake_count,
        'save_errors' : save_errors,
    }
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(data, f, indent=2)


def clear_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)

# HELPER — DISK SPACE CHECK

def get_folder_size(folder):
    total = 0
    for dirpath, _, filenames in os.walk(folder):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            try:
                total += os.path.getsize(fp)
            except OSError:
                pass
    return total


def check_disk_space_for_zip(folder):
    folder_size   = get_folder_size(folder)
    required      = int(folder_size * DISK_MARGIN_FACTOR)
    _, _, free    = shutil.disk_usage(BASE_DIR)

    folder_gb   = folder_size / (1024 ** 3)
    required_gb = required    / (1024 ** 3)
    free_gb     = free        / (1024 ** 3)

    print(f"  Folder size  : {folder_gb:.2f} GB")
    print(f"  Required free: {required_gb:.2f} GB  (folder × {DISK_MARGIN_FACTOR})")
    print(f"  Actual free  : {free_gb:.2f} GB")

    if free < required:
        raise RuntimeError(
            f"Insufficient disk space before zipping.\n"
            f"  Folder  : {folder_gb:.2f} GB\n"
            f"  Required: {required_gb:.2f} GB\n"
            f"  Free    : {free_gb:.2f} GB\n"
            f"Aborting to prevent archive corruption."
        )

# HELPER — ZIP, UPLOAD, VERIFY, CLEANUP

def create_zip(folder, zip_base_path):
    """Zip a single folder. Returns the .zip path."""
    print(f"  Archiving {os.path.basename(folder)}/  →  {os.path.basename(zip_base_path)}.zip")
    shutil.make_archive(zip_base_path, 'zip', os.path.dirname(folder), os.path.basename(folder))
    zip_path = zip_base_path + '.zip'

    if not os.path.exists(zip_path) or os.path.getsize(zip_path) == 0:
        raise RuntimeError(f"Zip creation failed or produced empty archive: {zip_path}")

    size_gb = os.path.getsize(zip_path) / (1024 ** 3)
    print(f"  Archive ready: {size_gb:.2f} GB")
    return zip_path


def upload_to_s3(local_path, s3_key):
    """Upload a file to S3."""
    size_gb = os.path.getsize(local_path) / (1024 ** 3)
    print(f"  Uploading {os.path.basename(local_path)}  ({size_gb:.2f} GB)  →  s3://{S3_BUCKET}/{s3_key}")
    s3.upload_file(local_path, S3_BUCKET, s3_key)


def verify_s3_upload(local_path, s3_key):
    """Confirm S3 object exists and its size matches local file."""
    local_size = os.path.getsize(local_path)
    try:
        response  = s3.head_object(Bucket=S3_BUCKET, Key=s3_key)
        s3_size   = response['ContentLength']
    except Exception as e:
        raise RuntimeError(f"S3 verification failed — head_object error: {e}")

    if s3_size != local_size:
        raise RuntimeError(
            f"S3 size mismatch for {s3_key}.\n"
            f"  Local : {local_size} bytes\n"
            f"  S3    : {s3_size} bytes\n"
            f"Upload may be incomplete. Local files preserved."
        )
    print(f"  Verified: S3 object size matches local ({s3_size / (1024**3):.2f} GB)")


def build_manifest(label, real_count, fake_count, save_errors,
                   error_samples, iteration_counter, zip_path, s3_key):
    count     = real_count if label == 'real' else fake_count
    size_gb   = os.path.getsize(zip_path) / (1024 ** 3) if os.path.exists(zip_path) else 0.0
    timestamp = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')

    error_block = ''
    if error_samples:
        error_block = '\nSampled error messages (first 10):\n'
        for i, msg in enumerate(error_samples, 1):
            error_block += f'  [{i:02d}] {msg}\n'

    return (
        f"OpenFake Production Download — {label.upper()} Class Manifest\n"
        f"{'=' * 60}\n"
        f"  Timestamp         : {timestamp}\n"
        f"  Class             : {label}\n"
        f"  Target per class  : {TARGET_PER_CLASS}\n"
        f"  Actual count      : {count}\n"
        f"  Save errors       : {save_errors}\n"
        f"  Iterations used   : {iteration_counter}\n"
        f"  Archive size      : {size_gb:.2f} GB\n"
        f"  S3 destination    : s3://{S3_BUCKET}/{s3_key}\n"
        f"{'=' * 60}\n"
        f"{error_block}"
    )


def process_class(label, folder, zip_base, manifest_path,
                  zip_s3_key, manifest_s3_key,
                  real_count, fake_count, save_errors,
                  error_samples, iteration_counter):
    """
    Full post-download pipeline for one class:
    disk check → zip → upload zip → verify → upload manifest → verify → cleanup
    """
    print(f"\n{'─' * 60}")
    print(f"Processing class: {label.upper()}")
    print(f"{'─' * 60}")

    # 1. Disk space check
    print("\n[1/5] Checking disk space...")
    check_disk_space_for_zip(folder)

    # 2. Zip
    print("\n[2/5] Creating archive...")
    zip_path = create_zip(folder, zip_base)

    # 3. Upload zip + verify
    print("\n[3/5] Uploading archive to S3...")
    try:
        upload_to_s3(zip_path, zip_s3_key)
        verify_s3_upload(zip_path, zip_s3_key)
    except Exception as e:
        print(f"\nUpload/verification failed — local files preserved.\n  {e}")
        raise

    # 4. Write + upload manifest
    print("\n[4/5] Writing and uploading manifest...")
    manifest_text = build_manifest(
        label, real_count, fake_count, save_errors,
        error_samples, iteration_counter, zip_path, zip_s3_key
    )
    with open(manifest_path, 'w') as f:
        f.write(manifest_text)
    print(manifest_text)

    try:
        upload_to_s3(manifest_path, manifest_s3_key)
        verify_s3_upload(manifest_path, manifest_s3_key)
    except Exception as e:
        print(f"\nManifest upload failed — local files preserved.\n  {e}")
        raise

    # 5. Cleanup this class only — only reached if both uploads verified
    print("\n[5/5] Cleaning up local artifacts for this class...")
    shutil.rmtree(folder, ignore_errors=True)
    os.remove(zip_path)
    os.remove(manifest_path)
    print(f"  Removed: {folder}")
    print(f"  Removed: {zip_path}")
    print(f"  Removed: {manifest_path}")

# SETUP

print("\n" + "═" * 60)
print("  OpenFake AWS Production Downloader")
print(f"  Target : {TARGET_PER_CLASS:,} real  +  {TARGET_PER_CLASS:,} fake  =  {TARGET_PER_CLASS * 2:,} total raw images")
print(f"  Bucket : s3://{S3_BUCKET}/{S3_PREFIX}/")
print("═" * 60 + "\n")

shutil.rmtree(TEMP_RAW_DIR, ignore_errors=True)
os.makedirs(REAL_DIR, exist_ok=True)
os.makedirs(FAKE_DIR, exist_ok=True)

# DATASET CONNECTION

print("[Setup] Connecting to ComplexDataLab/OpenFake stream...")
try:
    openfake = load_dataset(
        "ComplexDataLab/OpenFake",
        split='train',
        streaming=True,
        token=hf_token
    )
    openfake = openfake.decode(False)   # disable automatic image decoding
    print("[Setup] Connection successful.\n")
except Exception as e:
    raise RuntimeError(f"Dataset loading failed: {e}")

# DOWNLOAD LOOP

iteration_counter = 0
save_errors       = 0
error_samples     = []    # first 10 exception messages for manifest postmortem
real_count        = 0
fake_count        = 0
last_checkpoint   = 0     # tracks images saved since last checkpoint write

pbar_real = tqdm(total=TARGET_PER_CLASS, desc="  Real", unit="img", mininterval=1.0)
pbar_fake = tqdm(total=TARGET_PER_CLASS, desc="  Fake", unit="img", mininterval=1.0)

print("[Download] Starting single-pass stream...\n")

for item in openfake:
    iteration_counter += 1

    if iteration_counter > MAX_ITERATIONS:
        print("\n[Download] Iteration ceiling reached — stream stopped.")
        break

    image = None

    try:
        raw_label = item['label']
        raw_image = item.get('image', None)

        if raw_image is None:
            continue

        # -battle tested from sandbox-
        if isinstance(raw_label, int):
            label = 'real' if raw_label == 0 else 'fake'
        else:
            raw_label_str = str(raw_label).lower().strip()
            if 'real' in raw_label_str:
                label = 'real'
            elif any(x in raw_label_str for x in ['fake', 'sd', 'flux', 'mj', 'midjourney']):
                label = 'fake'
            else:
                continue

        # manual image decode so corrupt bytes can be skipped safely
        if isinstance(raw_image, dict):
            img_bytes = raw_image.get("bytes")
            img_path  = raw_image.get("path")

            if img_bytes is not None:
                image = PILImage.open(BytesIO(img_bytes))
            elif img_path:
                image = PILImage.open(img_path)
            else:
                continue
        else:
            image = raw_image

        image.load()

        if image.mode != 'RGB':
            image = image.convert('RGB')

        if label == 'real' and real_count < TARGET_PER_CLASS:
            filename = f"raw_openfake_real_{real_count:05d}.jpg"
            image.save(os.path.join(REAL_DIR, filename), format='JPEG', quality=95)
            real_count += 1
            pbar_real.update(1)

        elif label == 'fake' and fake_count < TARGET_PER_CLASS:
            filename = f"raw_openfake_fake_{fake_count:05d}.jpg"
            image.save(os.path.join(FAKE_DIR, filename), format='JPEG', quality=95)
            fake_count += 1
            pbar_fake.update(1)

    except (UnidentifiedImageError, OSError, ValueError) as e:
        save_errors += 1
        if len(error_samples) < 10:
            error_samples.append(f"iter={iteration_counter} | {type(e).__name__}: {str(e)[:120]}")
        continue

    except Exception as e:
        save_errors += 1
        if len(error_samples) < 10:
            error_samples.append(f"iter={iteration_counter} | {type(e).__name__}: {str(e)[:120]}")
        continue

    finally:
        try:
            if image is not None:
                image.close()
        except Exception:
            pass

    # -Periodic checkpoint-
    total_saved = real_count + fake_count
    if total_saved - last_checkpoint >= CHECKPOINT_EVERY:
        write_checkpoint(iteration_counter, real_count, fake_count, save_errors)
        last_checkpoint = total_saved

    if real_count == TARGET_PER_CLASS and fake_count == TARGET_PER_CLASS:
        break

pbar_real.close()
pbar_fake.close()

print(f"\n[Download] Complete.")
print(f"  Real      : {real_count:,} / {TARGET_PER_CLASS:,}")
print(f"  Fake      : {fake_count:,} / {TARGET_PER_CLASS:,}")
print(f"  Iterations: {iteration_counter:,}")
print(f"  Errors    : {save_errors}")

# FAIL HARD — do not proceed if targets not met

if real_count < TARGET_PER_CLASS or fake_count < TARGET_PER_CLASS:
    raise RuntimeError(
        f"\nTarget not reached — aborting. No zipping or uploading will occur.\n"
        f"  Real : {real_count:,} / {TARGET_PER_CLASS:,}\n"
        f"  Fake : {fake_count:,} / {TARGET_PER_CLASS:,}"
    )

clear_checkpoint()

# POST-DOWNLOAD PIPELINE — real class first, then fake

for label in ['real', 'fake']:
    process_class(
        label          = label,
        folder         = REAL_DIR         if label == 'real' else FAKE_DIR,
        zip_base       = LOCAL_ZIPS[label],
        manifest_path  = MANIFEST_PATHS[label],
        zip_s3_key     = S3_KEYS[f'{label}_zip'],
        manifest_s3_key= S3_KEYS[f'{label}_manifest'],
        real_count     = real_count,
        fake_count     = fake_count,
        save_errors    = save_errors,
        error_samples  = error_samples,
        iteration_counter = iteration_counter,
    )
clear_checkpoint()

# FINAL CLEANUP — base temp_raw only after both classes fully verified

shutil.rmtree(TEMP_RAW_DIR, ignore_errors=True)
print(f"\n[Cleanup] Removed base staging dir: {TEMP_RAW_DIR}")

print("\n" + "═" * 60)
print("  OpenFake production download — ALL DONE")
print(f"  Real zip : s3://{S3_BUCKET}/{S3_KEYS['real_zip']}")
print(f"  Fake zip : s3://{S3_BUCKET}/{S3_KEYS['fake_zip']}")
print("═" * 60 + "\n")


════════════════════════════════════════════════════════════
  OpenFake AWS Production Downloader
  Target : 50,000 real  +  50,000 fake  =  100,000 total raw images
  Bucket : s3://deepfake-d-100k-dataset-tw26/datasets/OpenFake/
════════════════════════════════════════════════════════════

[Setup] Connecting to ComplexDataLab/OpenFake stream...
[Setup] Connection successful.



  Real:   0%|          | 0/50000 [00:00<?, ?img/s]

  Fake:   0%|          | 0/50000 [00:00<?, ?img/s]

[Download] Starting single-pass stream...


[Download] Complete.
  Real      : 50,000 / 50,000
  Fake      : 50,000 / 50,000
  Iterations: 100,056
  Errors    : 1

────────────────────────────────────────────────────────────
Processing class: REAL
────────────────────────────────────────────────────────────

[1/5] Checking disk space...
  Folder size  : 6.18 GB
  Required free: 13.59 GB  (folder × 2.2)
  Actual free  : 447.37 GB

[2/5] Creating archive...
  Archiving real/  →  openfake_real_raw.zip
  Archive ready: 5.88 GB

[3/5] Uploading archive to S3...
  Uploading openfake_real_raw.zip  (5.88 GB)  →  s3://deepfake-d-100k-dataset-tw26/datasets/OpenFake/openfake_real_raw.zip
  Verified: S3 object size matches local (5.88 GB)

[4/5] Writing and uploading manifest...
OpenFake Production Download — REAL Class Manifest
  Timestamp         : 2026-03-23 14:07:19 UTC
  Class             : real
  Target per class  : 50000
  Actual count      : 50000
  Save errors       : 1
  Iterations used 

Cleanup Cell if the Download gets Interrupted.

In [7]:
import os, shutil

base_dir = os.environ.get("OPENFAKE_BASE_DIR", "/home/ec2-user/SageMaker")
temp_raw_dir = os.path.join(base_dir, "temp_raw")

if os.path.exists(temp_raw_dir):
    shutil.rmtree(temp_raw_dir, ignore_errors=True)
    print(f"Removed: {temp_raw_dir}")
else:
    print("No temp_raw directory found.")

# optional: also remove any leftover local zips/manifests/checkpoints if your script uses them
for name in [
    "openfake_real_raw.zip",
    "openfake_fake_raw.zip",
    "openfake_checkpoint.json",
    "openfake_real_manifest.txt",
    "openfake_fake_manifest.txt",
]:
    path = os.path.join(base_dir, name)
    if os.path.exists(path):
        os.remove(path)
        print(f"Removed: {path}")

Removed: /home/ec2-user/SageMaker/temp_raw


RetinaFace Extractor on OpenFake Raw 100K Images. 

Installers.

In [2]:
%pip install -U ImageHash pybktree retina-face

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.2/572.2 MB 57.9 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 115.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 160.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 121.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 133.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 175.2 MB/s  0:00:00
  Created wheel for pybktree: filename=pybktree-1.1-py3-none-any.whl size=5025 sha256=5be9553f7f0c553bafbaf61ca30756d1663f9e11633fa7d8284198600922102d
  Stored in directory: /home/ec2-user/.cache/pip/wheels/09/97/f5/14ae07459879c2738cfa34f61a8d3d2d99b13e15328bc6d1dc
Successfully built pybktree
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18/18 [retina-face] [tensorflow]
Note: you may need to res

In [2]:
%pip install -U tf-keras

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 97.1 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


Preflight Check.

In [1]:
import cv2
import numpy as np
import imagehash
import pybktree
import boto3
from retinaface import RetinaFace
from PIL import Image
from tqdm.auto import tqdm

print("All critical imports OK")

I0000 00:00:1774337996.244929    8168 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


All critical imports OK


Script.

In [2]:
# RetinaFace Bouncer — AWS SageMaker Production.
# Architecture:
#   - Pull real and fake raw zips separately from S3
#   - Extract to local SSD
#   - Run RetinaFace face extraction + deduplication in one pass
#   - Zip processed faces - upload to S3
#   - Upload CSV log and manifest to S3
#   - Clean up local workspace only after all uploads verified

import os
import cv2
import numpy as np
import shutil
import glob
import csv
import imagehash
import random
import pybktree
import boto3
import warnings
from datetime import datetime, timezone
from retinaface import RetinaFace
from PIL import Image
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

random.seed(42)
np.random.seed(42)

# CONFIGURATION  —  change these, everything else derives from them

BASE_DIR  = os.environ.get('BOUNCER_BASE_DIR', '/home/ec2-user/SageMaker')

S3_BUCKET = 'deepfake-d-100k-dataset-tw26'

# Input zips (produced by the downloader)
S3_INPUT_KEYS = {
    'real' : 'datasets/OpenFake/openfake_real_raw.zip',
    'fake' : 'datasets/OpenFake/openfake_fake_raw.zip',
}

# Output destinations
S3_OUTPUT_PREFIX = 'datasets/OpenFake/processed'
S3_OUTPUT_KEYS = {
    'faces_zip' : f'{S3_OUTPUT_PREFIX}/openfake_processed_faces.zip',
    'csv_log'   : f'{S3_OUTPUT_PREFIX}/openfake_face_extraction_log.csv',
    'manifest'  : f'{S3_OUTPUT_PREFIX}/openfake_bouncer_manifest.txt',
}

DISK_MARGIN_FACTOR = 2.2    # require 2.2× folder size free before zipping

# Quota system (70 / 15 / 15 split)
# OpenFake = 40% of 100K total = 40K processed faces (20K real + 20K fake)
target_quotas = {
    "train_real" : 14000, "val_real" : 3000, "test_real" : 3000,
    "train_fake" : 14000, "val_fake" : 3000, "test_fake" : 3000,
}

# DERIVED PATHS

TEMP_WORKSPACE  = os.path.join(BASE_DIR, 'temp_workspace')
LOCAL_INPUT     = os.path.join(TEMP_WORKSPACE, 'input_frames')
LOCAL_OUTPUT    = os.path.join(TEMP_WORKSPACE, 'processed_faces')
LOCAL_ZIP_OUT   = os.path.join(TEMP_WORKSPACE, 'openfake_processed_faces')   # .zip appended
CSV_LOG_PATH    = os.path.join(TEMP_WORKSPACE, 'openfake_face_extraction_log.csv')
MANIFEST_PATH   = os.path.join(TEMP_WORKSPACE, 'openfake_bouncer_manifest.txt')

# S3 CLIENT

s3 = boto3.client('s3')

# HELPERS — DISK

def get_folder_size(folder):
    total = 0
    for dirpath, _, filenames in os.walk(folder):
        for f in filenames:
            try:
                total += os.path.getsize(os.path.join(dirpath, f))
            except OSError:
                pass
    return total


def check_disk_space_for_zip(folder):
    folder_size = get_folder_size(folder)
    required    = int(folder_size * DISK_MARGIN_FACTOR)
    _, _, free  = shutil.disk_usage(BASE_DIR)

    folder_gb   = folder_size / (1024 ** 3)
    required_gb = required    / (1024 ** 3)
    free_gb     = free        / (1024 ** 3)

    print(f"  Folder size  : {folder_gb:.2f} GB")
    print(f"  Required free: {required_gb:.2f} GB  (folder × {DISK_MARGIN_FACTOR})")
    print(f"  Actual free  : {free_gb:.2f} GB")

    if free < required:
        raise RuntimeError(
            f"Insufficient disk space before zipping.\n"
            f"  Folder  : {folder_gb:.2f} GB\n"
            f"  Required: {required_gb:.2f} GB\n"
            f"  Free    : {free_gb:.2f} GB\n"
            f"Aborting to prevent archive corruption."
        )

# HELPERS — S3

def download_from_s3(s3_key, local_path):
    size_info = s3.head_object(Bucket=S3_BUCKET, Key=s3_key)['ContentLength']
    print(f"  Downloading s3://{S3_BUCKET}/{s3_key}  ({size_info / (1024**3):.2f} GB)...")
    s3.download_file(S3_BUCKET, s3_key, local_path)
    if not os.path.exists(local_path) or os.path.getsize(local_path) == 0:
        raise RuntimeError(f"Download failed or produced empty file: {local_path}")
    print(f"  Downloaded → {local_path}")


def upload_to_s3(local_path, s3_key):
    size_gb = os.path.getsize(local_path) / (1024 ** 3)
    print(f"  Uploading {os.path.basename(local_path)}  ({size_gb:.2f} GB)  →  s3://{S3_BUCKET}/{s3_key}")
    s3.upload_file(local_path, S3_BUCKET, s3_key)


def verify_s3_upload(local_path, s3_key):
    local_size = os.path.getsize(local_path)
    try:
        s3_size = s3.head_object(Bucket=S3_BUCKET, Key=s3_key)['ContentLength']
    except Exception as e:
        raise RuntimeError(f"S3 verification failed — head_object error: {e}")
    if s3_size != local_size:
        raise RuntimeError(
            f"S3 size mismatch for {s3_key}.\n"
            f"  Local : {local_size} bytes\n"
            f"  S3    : {s3_size} bytes\n"
            f"Upload may be incomplete. Local files preserved."
        )
    print(f"  Verified: S3 size matches local ({s3_size / (1024**3):.2f} GB)")

# HELPERS — QUOTA ROUTER (unchanged from sandbox)

accepted_counts = {key: 0 for key in target_quotas}
total_rejected  = 0


def get_target_bucket(category):
    """Dynamically routes accepted faces to fill Train → Val → Test sequentially."""
    for split in ["train", "val", "test"]:
        bucket = f"{split}_{category}"
        if accepted_counts[bucket] < target_quotas[bucket]:
            return bucket, split
    return None, None

# MASTER CROP ENGINE (unchanged from sandbox — battle tested)

def hash_distance(hash1, hash2):
    return hash1 - hash2

global_seen_tree = pybktree.BKTree(hash_distance)


def master_crop_engine(img_path, save_path,
                       padding=25, min_face_size=30,
                       min_face_ratio=0.005, blur_threshold=50,
                       confidence_threshold=0.90):
    try:
        img_cv = cv2.imread(img_path)
        if img_cv is None:
            return False, "Corrupted Image", 0.0, 0.0, "", "[]"

        height, width = img_cv.shape[:2]
        faces = RetinaFace.detect_faces(img_cv)

        if not isinstance(faces, dict) or len(faces) == 0:
            return False, "No Face Detected", 0.0, 0.0, "", "[]"

        largest_area = 0
        best_face    = None
        for face in faces.values():
            box = face.get('facial_area', None)
            if box is None:
                continue
            area = (box[2] - box[0]) * (box[3] - box[1])
            if area > largest_area:
                largest_area = area
                best_face    = face

        if best_face is None:
            return False, "No Valid Face Box", 0.0, 0.0, "", "[]"

        box        = best_face['facial_area']
        str_box    = f"[{box[0]}, {box[1]}, {box[2]}, {box[3]}]"
        confidence = best_face['score']

        if confidence < confidence_threshold:
            return False, "Low Confidence", confidence, 0.0, "", str_box

        face_w, face_h = box[2] - box[0], box[3] - box[1]
        if face_w < min_face_size or face_h < min_face_size:
            return False, "Resolution Too Small", confidence, 0.0, "", str_box
        if (face_w * face_h) / (width * height) < min_face_ratio:
            return False, "Face Ratio Too Small", confidence, 0.0, "", str_box

        x_min = max(0, int(box[0]) - padding)
        y_min = max(0, int(box[1]) - padding)
        x_max = min(width,  int(box[2]) + padding)
        y_max = min(height, int(box[3]) + padding)
        cropped_cv = img_cv[y_min:y_max, x_min:x_max]

        gray_crop = cv2.cvtColor(cropped_cv, cv2.COLOR_BGR2GRAY)
        blur_val  = cv2.Laplacian(gray_crop, cv2.CV_64F).var()

        if blur_val < blur_threshold:
            return False, "Motion Blur", confidence, blur_val, "", str_box

        cropped_pil    = Image.fromarray(cv2.cvtColor(cropped_cv, cv2.COLOR_BGR2RGB))
        standardized   = cropped_pil.resize((260, 260), Image.BICUBIC)
        new_hash       = imagehash.phash(standardized)

        # Global cross-split leakage check
        matches = global_seen_tree.find(new_hash, 2)
        if matches:
            return False, "Global Duplicate Face (Leakage Prevented)", confidence, blur_val, str(new_hash), str_box

        global_seen_tree.add(new_hash)
        standardized.save(save_path, format='JPEG', quality=95)
        return True, "Accepted", confidence, blur_val, str(new_hash), str_box

    except Exception as e:
        return False, f"Engine Error: {str(e)}", 0.0, 0.0, "", "[]"

# SETUP

print("\n" + "═" * 60)
print("  RetinaFace Bouncer — AWS SageMaker Production")
print(f"  Dataset  : OpenFake")
print(f"  Target   : {sum(target_quotas.values()):,} processed faces  "
      f"({sum(v for k,v in target_quotas.items() if 'real' in k):,} real  +  "
      f"{sum(v for k,v in target_quotas.items() if 'fake' in k):,} fake)")
print(f"  Bucket   : s3://{S3_BUCKET}")
print("═" * 60 + "\n")

shutil.rmtree(TEMP_WORKSPACE, ignore_errors=True)
os.makedirs(LOCAL_INPUT,  exist_ok=True)
os.makedirs(LOCAL_OUTPUT, exist_ok=True)

# CSV log header
with open(CSV_LOG_PATH, mode='w', newline='') as f:
    csv.writer(f).writerow([
        "Image_Name", "Assigned_Split", "Category",
        "Status", "Reason", "Confidence",
        "Blur_Score", "pHash", "Bounding_Box"
    ])

# PULL INPUT ZIPS FROM S3

print("[Setup] Pulling raw image zips from S3...")

for label, s3_key in S3_INPUT_KEYS.items():
    local_zip = os.path.join(TEMP_WORKSPACE, f'openfake_{label}_raw.zip')
    download_from_s3(s3_key, local_zip)

    print(f"  Unpacking {label} zip...")
    shutil.unpack_archive(local_zip, LOCAL_INPUT)
    os.remove(local_zip)
    print(f"  {label} unpacked and zip removed.\n")

# Post-unpack structure validation
for required_dir in [os.path.join(LOCAL_INPUT, 'real'), os.path.join(LOCAL_INPUT, 'fake')]:
    if not os.path.isdir(required_dir):
        raise RuntimeError(
            f"Expected directory not found after unpack: {required_dir}\n"
            f"Check that the zip archives contain a top-level 'real/' and 'fake/' folder."
        )
print("[Setup] Input structure validated — real/ and fake/ directories confirmed.\n")

# FACE EXTRACTION LOOP

print("[Bouncer] Firing up RetinaFace Deduplication Engine...")

image_files = glob.glob(os.path.join(LOCAL_INPUT, "**", "*.jpg"), recursive=True)

if not image_files:
    raise ValueError("CRITICAL: No images found after unpacking. Check zip structure.")

print(f"[Bouncer] Found {len(image_files):,} raw images. Shuffling to prevent split bias...\n")
random.shuffle(image_files)

pbar = tqdm(
    total=sum(target_quotas.values()),
    desc="Securing OpenFake Quota",
    unit="face"
)

run_start = datetime.now(timezone.utc)

with open(CSV_LOG_PATH, mode='a', newline='') as log_file:
    csv_writer = csv.writer(log_file)

    for img_path in image_files:
        cat_name = os.path.basename(os.path.dirname(img_path)).lower().strip()

        if cat_name not in ["real", "fake"]:
            continue

        bucket_key, assigned_split = get_target_bucket(cat_name)
        if not bucket_key:
            continue

        out_folder = os.path.join(LOCAL_OUTPUT, assigned_split, cat_name)
        os.makedirs(out_folder, exist_ok=True)

        save_path = os.path.join(out_folder, os.path.basename(img_path))

        # Collision protection — do not silently overwrite an existing output file
        if os.path.exists(save_path):
            total_rejected += 1
            csv_writer.writerow([
                os.path.basename(img_path), assigned_split, cat_name,
                "Rejected", "Output Filename Collision",
                0.0, 0.0, "", "[]"
            ])
            continue

        passed, msg, conf, blur, phash_val, bbox = master_crop_engine(img_path, save_path)

        if passed:
            accepted_counts[bucket_key] += 1
            csv_writer.writerow([
                os.path.basename(img_path), assigned_split, cat_name,
                "Accepted", "None",
                round(conf, 4), round(blur, 2), phash_val, bbox
            ])
            pbar.update(1)
        else:
            total_rejected += 1
            log_split = assigned_split if "Duplicate" not in msg else "Unassigned"
            csv_writer.writerow([
                os.path.basename(img_path), log_split, cat_name,
                "Rejected", msg,
                round(conf, 4), round(blur, 2), phash_val, bbox
            ])

        if all(accepted_counts[k] >= target_quotas[k] for k in target_quotas):
            print("\n[Bouncer] All target quotas achieved. Shutting down.")
            break

pbar.close()

total_secured = sum(accepted_counts.values())
run_end       = datetime.now(timezone.utc)
duration_min  = (run_end - run_start).total_seconds() / 60

print(f"\n[Bouncer] Extraction complete.")
print(f"  Secured : {total_secured:,}")
print(f"  Rejected: {total_rejected:,}")
print(f"  Duration: {duration_min:.1f} minutes\n")

# Quota breakdown
print("  Split breakdown:")
for key, count in accepted_counts.items():
    quota = target_quotas[key]
    print(f"    {key:<15} {count:>6,} / {quota:,}")

# FAIL HARD IF QUOTAS NOT MET

shortfalls = {k: target_quotas[k] - accepted_counts[k]
              for k in target_quotas if accepted_counts[k] < target_quotas[k]}

if shortfalls:
    raise RuntimeError(
        f"\nQuota not reached — aborting. No upload will occur.\n"
        + "\n".join(f"  {k}: {accepted_counts[k]:,} / {target_quotas[k]:,}  (short by {v:,})"
                    for k, v in shortfalls.items())
    )

# ZIP OUTPUT

print("\n[Upload] Checking disk space before archiving...")
check_disk_space_for_zip(LOCAL_OUTPUT)

print("[Upload] Archiving processed faces...")
shutil.make_archive(LOCAL_ZIP_OUT, 'zip', LOCAL_OUTPUT)

zip_path = LOCAL_ZIP_OUT + '.zip'
if not os.path.exists(zip_path) or os.path.getsize(zip_path) == 0:
    raise RuntimeError("Zip creation failed or produced empty archive.")

zip_size_gb = os.path.getsize(zip_path) / (1024 ** 3)
print(f"  Archive ready: {zip_size_gb:.2f} GB")

# MANIFEST

manifest_text = (
    f"RetinaFace Bouncer — OpenFake Production Manifest\n"
    f"{'=' * 60}\n"
    f"  Timestamp         : {run_end.strftime('%Y-%m-%d %H:%M:%S UTC')}\n"
    f"  Duration          : {duration_min:.1f} minutes\n"
    f"  Total secured     : {total_secured:,}\n"
    f"  Total rejected    : {total_rejected:,}\n"
    f"  Pass rate         : {total_secured / max(total_secured + total_rejected, 1) * 100:.1f}%\n"
    f"  Archive size      : {zip_size_gb:.2f} GB\n"
    f"  S3 output         : s3://{S3_BUCKET}/{S3_OUTPUT_KEYS['faces_zip']}\n"
    f"{'=' * 60}\n"
    f"  Split breakdown:\n"
    + "".join(f"    {k:<15} {accepted_counts[k]:>6,} / {target_quotas[k]:,}\n"
              for k in target_quotas)
)

with open(MANIFEST_PATH, 'w') as f:
    f.write(manifest_text)
print("\n" + manifest_text)

# UPLOAD ALL OUTPUTS TO S3

upload_success = False
try:
    print("[Upload] Uploading processed faces zip...")
    upload_to_s3(zip_path, S3_OUTPUT_KEYS['faces_zip'])
    verify_s3_upload(zip_path, S3_OUTPUT_KEYS['faces_zip'])

    print("\n[Upload] Uploading CSV log...")
    upload_to_s3(CSV_LOG_PATH, S3_OUTPUT_KEYS['csv_log'])
    verify_s3_upload(CSV_LOG_PATH, S3_OUTPUT_KEYS['csv_log'])

    print("\n[Upload] Uploading manifest...")
    upload_to_s3(MANIFEST_PATH, S3_OUTPUT_KEYS['manifest'])
    verify_s3_upload(MANIFEST_PATH, S3_OUTPUT_KEYS['manifest'])

    upload_success = True

except Exception as e:
    print(f"\nUpload failed — local files preserved. Do not shut down instance.\n  {e}")
    raise

# CLEANUP — only after all uploads verified

if upload_success:
    shutil.rmtree(TEMP_WORKSPACE, ignore_errors=True)
    print(f"\n[Cleanup] Workspace removed: {TEMP_WORKSPACE}")

print("\n" + "═" * 60)
print("  RetinaFace Bouncer — ALL DONE")
print(f"  Faces  : s3://{S3_BUCKET}/{S3_OUTPUT_KEYS['faces_zip']}")
print(f"  Log    : s3://{S3_BUCKET}/{S3_OUTPUT_KEYS['csv_log']}")
print(f"  Report : s3://{S3_BUCKET}/{S3_OUTPUT_KEYS['manifest']}")
print("═" * 60 + "\n")


════════════════════════════════════════════════════════════
  RetinaFace Bouncer — AWS SageMaker Production
  Dataset  : OpenFake
  Target   : 40,000 processed faces  (20,000 real  +  20,000 fake)
  Bucket   : s3://deepfake-d-100k-dataset-tw26
════════════════════════════════════════════════════════════

[Setup] Pulling raw image zips from S3...
  Downloaded → /home/ec2-user/SageMaker/temp_workspace/openfake_real_raw.zip
  Unpacking real zip...
  real unpacked and zip removed.

  Downloaded → /home/ec2-user/SageMaker/temp_workspace/openfake_fake_raw.zip
  Unpacking fake zip...
  fake unpacked and zip removed.

[Setup] Input structure validated — real/ and fake/ directories confirmed.

[Bouncer] Firing up RetinaFace Deduplication Engine...
[Bouncer] Found 100,000 raw images. Shuffling to prevent split bias...



Securing OpenFake Quota:   0%|          | 0/40000 [00:00<?, ?face/s]

W0000 00:00:1774340140.143984    8168 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
I0000 00:00:1774340140.145244    8168 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 20833 MB memory:  -> device: 0, name: NVIDIA L4, pci bus id: 0000:31:00.0, compute capability: 8.9
Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5
To: /home/ec2-user/.deepface/weights/retinaface.h5


26-03-24 08:15:41 - Directory /home/ec2-user/.deepface created
26-03-24 08:15:41 - Directory /home/ec2-user/.deepface/weights created
26-03-24 08:15:41 - retinaface.h5 will be downloaded from the url https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5



  0%|          | 0.00/119M [00:00<?, ?B/s]
 40%|███▉      | 47.2M/119M [00:00<00:00, 471MB/s]
100%|██████████| 119M/119M [00:00<00:00, 478MB/s] 
I0000 00:00:1774340143.297507   15489 cuda_dnn.cc:461] Loaded cuDNN version 91002



[Bouncer] All target quotas achieved. Shutting down.

[Bouncer] Extraction complete.
  Secured : 40,000
  Rejected: 25,909
  Duration: 99.0 minutes

  Split breakdown:
    train_real      14,000 / 14,000
    val_real         3,000 / 3,000
    test_real        3,000 / 3,000
    train_fake      14,000 / 14,000
    val_fake         3,000 / 3,000
    test_fake        3,000 / 3,000

[Upload] Checking disk space before archiving...
  Folder size  : 0.83 GB
  Required free: 1.83 GB  (folder × 2.2)
  Actual free  : 446.45 GB
[Upload] Archiving processed faces...
  Archive ready: 0.83 GB

RetinaFace Bouncer — OpenFake Production Manifest
  Timestamp         : 2026-03-24 09:54:28 UTC
  Duration          : 99.0 minutes
  Total secured     : 40,000
  Total rejected    : 25,909
  Pass rate         : 60.7%
  Archive size      : 0.83 GB
  S3 output         : s3://deepfake-d-100k-dataset-tw26/datasets/OpenFake/processed/openfake_processed_faces.zip
  Split breakdown:
    train_real      14,000 / 14,0

Download Script for FF++.

TUM API Setup.

In [4]:
# FF++ Video Downloader — AWS SageMaker Production

# Architecture:
#   - Pulls TUM download script using URL from environment variable
#   - Downloads real and fake videos per category to local SSD
#   - Zips real and fake separately → uploads to S3 → verifies → cleans up
#   - Separate manifests per class uploaded to S3
#   - base temp dir deleted only after both uploads are fully verified

import os
import subprocess
import shutil
import glob
import boto3
import urllib.request
from datetime import datetime, timezone
from tqdm.auto import tqdm

# CONFIGURATION.

BASE_DIR = os.environ.get('FFPP_BASE_DIR', '/home/ec2-user/SageMaker')

S3_BUCKET = 'deepfake-d-100k-dataset-tw26'
S3_PREFIX = 'datasets/FFPlus'

DISK_MARGIN_FACTOR = 2.2    # require 2.2× folder size free before zipping

# ── Download targets ──────────────────────────────────────────────────────────
download_targets = {
    'original'       : 1000,   # real videos
    'Deepfakes'      : 1000,   # fake subcategory
    'Face2Face'      : 1000,   # fake subcategory
    'FaceSwap'       : 1000,   # fake subcategory
    'NeuralTextures' : 1000,   # fake subcategory
}

DOWNLOAD_SERVER  = 'EU2'
COMPRESSION      = 'c23'

# DERIVED PATHS

TEMP_BASE           = os.path.join(BASE_DIR, 'ffpp_temp')
REAL_VIDEO_DIR      = os.path.join(TEMP_BASE, 'real')
FAKE_VIDEO_DIR      = os.path.join(TEMP_BASE, 'fake')
DOWNLOAD_STAGE_BASE = os.path.join(TEMP_BASE, 'download_stage')   # staging only — never zipped
DOWNLOAD_SCRIPT     = os.path.join(BASE_DIR,  'ffpp_download.py')

LOCAL_ZIPS = {
    'real' : os.path.join(BASE_DIR, 'ffpp_real_videos'),   # .zip appended by make_archive
    'fake' : os.path.join(BASE_DIR, 'ffpp_fake_videos'),
}

MANIFEST_PATHS = {
    'real' : os.path.join(BASE_DIR, 'ffpp_real_manifest.txt'),
    'fake' : os.path.join(BASE_DIR, 'ffpp_fake_manifest.txt'),
}

S3_KEYS = {
    'real_zip'      : f'{S3_PREFIX}/ffpp_real_videos.zip',
    'fake_zip'      : f'{S3_PREFIX}/ffpp_fake_videos.zip',
    'real_manifest' : f'{S3_PREFIX}/ffpp_real_manifest.txt',
    'fake_manifest' : f'{S3_PREFIX}/ffpp_fake_manifest.txt',
}

# S3 CLIENT

s3 = boto3.client('s3')

# HELPERS — DISK

def get_folder_size(folder):
    total = 0
    for dirpath, _, filenames in os.walk(folder):
        for f in filenames:
            try:
                total += os.path.getsize(os.path.join(dirpath, f))
            except OSError:
                pass
    return total


def check_disk_space_for_zip(folder):
    folder_size = get_folder_size(folder)
    required    = int(folder_size * DISK_MARGIN_FACTOR)
    _, _, free  = shutil.disk_usage(BASE_DIR)

    folder_gb   = folder_size / (1024 ** 3)
    required_gb = required    / (1024 ** 3)
    free_gb     = free        / (1024 ** 3)

    print(f"  Folder size  : {folder_gb:.2f} GB")
    print(f"  Required free: {required_gb:.2f} GB  (folder × {DISK_MARGIN_FACTOR})")
    print(f"  Actual free  : {free_gb:.2f} GB")

    if free < required:
        raise RuntimeError(
            f"Insufficient disk space before zipping.\n"
            f"  Folder  : {folder_gb:.2f} GB\n"
            f"  Required: {required_gb:.2f} GB\n"
            f"  Free    : {free_gb:.2f} GB\n"
            f"Aborting to prevent archive corruption."
        )

# HELPERS — S3

def upload_to_s3(local_path, s3_key):
    size_gb = os.path.getsize(local_path) / (1024 ** 3)
    print(f"  Uploading {os.path.basename(local_path)}  ({size_gb:.2f} GB)  →  s3://{S3_BUCKET}/{s3_key}")
    s3.upload_file(local_path, S3_BUCKET, s3_key)


def verify_s3_upload(local_path, s3_key):
    local_size = os.path.getsize(local_path)
    try:
        s3_size = s3.head_object(Bucket=S3_BUCKET, Key=s3_key)['ContentLength']
    except Exception as e:
        raise RuntimeError(f"S3 verification failed — head_object error: {e}")
    if s3_size != local_size:
        raise RuntimeError(
            f"S3 size mismatch for {s3_key}.\n"
            f"  Local : {local_size} bytes\n"
            f"  S3    : {s3_size} bytes\n"
            f"Upload may be incomplete. Local files preserved."
        )
    print(f"  Verified: S3 size matches local ({s3_size / (1024**3):.2f} GB)")

# HELPERS — ZIP + UPLOAD PIPELINE

def create_zip(folder, zip_base_path):
    print(f"  Archiving {os.path.basename(folder)}/  →  {os.path.basename(zip_base_path)}.zip")
    shutil.make_archive(zip_base_path, 'zip', os.path.dirname(folder), os.path.basename(folder))
    zip_path = zip_base_path + '.zip'
    if not os.path.exists(zip_path) or os.path.getsize(zip_path) == 0:
        raise RuntimeError(f"Zip creation failed or produced empty archive: {zip_path}")
    size_gb = os.path.getsize(zip_path) / (1024 ** 3)
    print(f"  Archive ready: {size_gb:.2f} GB")
    return zip_path


def build_manifest(label, video_counts, total_videos, zip_path, s3_key):
    size_gb   = os.path.getsize(zip_path) / (1024 ** 3) if os.path.exists(zip_path) else 0.0
    timestamp = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')

    if label == 'real':
        breakdown = f"  original  : {video_counts.get('original', 0):>6,} videos\n"
    else:
        breakdown = "".join(
            f"  {cat:<16}: {video_counts.get(cat, 0):>6,} videos\n"
            for cat in ['Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures']
        )

    return (
        f"FF++ Video Downloader — {label.upper()} Class Manifest\n"
        f"{'=' * 60}\n"
        f"  Timestamp         : {timestamp}\n"
        f"  Class             : {label}\n"
        f"  Total videos      : {total_videos:,}\n"
        f"  Archive size      : {size_gb:.2f} GB\n"
        f"  S3 destination    : s3://{S3_BUCKET}/{s3_key}\n"
        f"{'=' * 60}\n"
        f"  Category breakdown:\n"
        f"{breakdown}"
    )


def process_class_upload(label, folder, zip_base, manifest_path,
                         zip_s3_key, manifest_s3_key,
                         video_counts, total_videos):
    """Disk check → zip → upload → verify → manifest → upload → verify → cleanup."""
    print(f"\n{'─' * 60}")
    print(f"Processing class: {label.upper()}")
    print(f"{'─' * 60}")

    print("\n[1/5] Checking disk space...")
    check_disk_space_for_zip(folder)

    print("\n[2/5] Creating archive...")
    zip_path = create_zip(folder, zip_base)

    print("\n[3/5] Uploading archive to S3...")
    try:
        upload_to_s3(zip_path, zip_s3_key)
        verify_s3_upload(zip_path, zip_s3_key)
    except Exception as e:
        print(f"\nUpload/verification failed — local files preserved.\n  {e}")
        raise

    print("\n[4/5] Writing and uploading manifest...")
    manifest_text = build_manifest(label, video_counts, total_videos, zip_path, zip_s3_key)
    with open(manifest_path, 'w') as f:
        f.write(manifest_text)
    print(manifest_text)

    try:
        upload_to_s3(manifest_path, manifest_s3_key)
        verify_s3_upload(manifest_path, manifest_s3_key)
    except Exception as e:
        print(f"\nManifest upload failed — local files preserved.\n  {e}")
        raise

    print("\n[5/5] Cleaning up local artifacts for this class...")
    shutil.rmtree(folder, ignore_errors=True)
    os.remove(zip_path)
    os.remove(manifest_path)
    print(f"  Removed: {folder}")
    print(f"  Removed: {zip_path}")

# SETUP

print("\n" + "═" * 60)
print("  FF++ Video Downloader — AWS SageMaker Production")
total_videos_planned = sum(download_targets.values())
print(f"  Target   : {total_videos_planned:,} videos  "
      f"(1,000 real  +  4,000 fake across 4 subcategories)")
print(f"  Bucket   : s3://{S3_BUCKET}/{S3_PREFIX}/")
print("═" * 60 + "\n")

# Pull TUM URL from environment
TUM_URL = os.environ.get('TUM_LINK')
if not TUM_URL:
    raise RuntimeError(
        "TUM_LINK environment variable is not set.\n"
        "Run the API setup cell first."
    )

# Workspace setup
shutil.rmtree(TEMP_BASE, ignore_errors=True)
os.makedirs(REAL_VIDEO_DIR,      exist_ok=True)
os.makedirs(FAKE_VIDEO_DIR,      exist_ok=True)
os.makedirs(DOWNLOAD_STAGE_BASE, exist_ok=True)

# Fetch TUM download script
print("[Setup] Fetching TUM download script...")
urllib.request.urlretrieve(TUM_URL, DOWNLOAD_SCRIPT)
if not os.path.exists(DOWNLOAD_SCRIPT) or os.path.getsize(DOWNLOAD_SCRIPT) == 0:
    raise RuntimeError("Failed to fetch TUM download script.")
print(f"  Saved → {DOWNLOAD_SCRIPT}\n")

# DOWNLOAD LOOP

video_counts    = {}   # tracks actual videos secured per category
download_errors = []   # categories that fell short of target

print("[Download] Starting FF++ video download...\n")

for category, num_videos in download_targets.items():

    is_real    = (category == 'original')
    sub_folder = 'real' if is_real else f'fake/{category.lower()}'
    temp_path  = os.path.join(DOWNLOAD_STAGE_BASE, sub_folder)

    shutil.rmtree(temp_path, ignore_errors=True)
    os.makedirs(temp_path, exist_ok=True)

    print(f"[Download] Pulling {num_videos:,} videos — {category}...")
    cmd = (
        f'echo "" | python3 {DOWNLOAD_SCRIPT} {temp_path} '
        f'-d {category} -c {COMPRESSION} -n {num_videos} --server {DOWNLOAD_SERVER}'
    )
    subprocess.run(cmd, shell=True)

    # Recursive search — bypasses TUM's nested folder structure
    local_vids = glob.glob(os.path.join(temp_path, "**", "*.mp4"), recursive=True)
    secured    = len(local_vids)
    video_counts[category] = secured

    print(f"  Secured: {secured:,} / {num_videos:,} videos for {category}")

    if secured < num_videos:
        download_errors.append(
            f"{category}: {secured:,} / {num_videos:,}  (short by {num_videos - secured:,})"
        )

    # Move videos into the correct class folder
    dest_base = REAL_VIDEO_DIR if is_real else os.path.join(FAKE_VIDEO_DIR, category.lower())
    os.makedirs(dest_base, exist_ok=True)

    for vid_path in tqdm(local_vids, desc=f"  Moving {category}", unit="vid"):
        shutil.move(vid_path, os.path.join(dest_base, os.path.basename(vid_path)))

    shutil.rmtree(temp_path, ignore_errors=True)
    print()

# Download summary
total_real_secured = video_counts.get('original', 0)
total_fake_secured = sum(video_counts.get(c, 0)
                         for c in ['Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures'])
total_secured      = total_real_secured + total_fake_secured

print(f"[Download] Complete.")
print(f"  Real  : {total_real_secured:,} videos")
print(f"  Fake  : {total_fake_secured:,} videos")
print(f"  Total : {total_secured:,} videos")

if download_errors:
    print(f"\n  Shortfalls detected:")
    for e in download_errors:
        print(f"    {e}")

# Fail hard if either class is empty
if total_real_secured == 0 or total_fake_secured == 0:
    raise RuntimeError(
        f"Critical failure — one or both classes have zero videos.\n"
        f"  Real : {total_real_secured:,}\n"
        f"  Fake : {total_fake_secured:,}\n"
        f"Aborting — no zip or upload will occur."
    )

# Strict per-category quota enforcement
quota_failures = []
for category, target in download_targets.items():
    secured = video_counts.get(category, 0)
    if secured < target:
        quota_failures.append(
            f"  {category}: secured {secured:,} / {target:,}  (short by {target - secured:,})"
        )

if quota_failures:
    failure_lines = "\n".join(quota_failures)
    raise RuntimeError(
        f"Per-category quota check failed — the following categories are under target:\n"
        f"{failure_lines}\n"
        f"Aborting — no zip or upload will occur."
    )

# Verify final class folders survived the download loop
if not os.path.isdir(REAL_VIDEO_DIR):
    raise RuntimeError(
        f"REAL_VIDEO_DIR missing before zip pipeline — expected: {REAL_VIDEO_DIR}\n"
        f"Aborting — no zip or upload will occur."
    )
if not os.path.isdir(FAKE_VIDEO_DIR):
    raise RuntimeError(
        f"FAKE_VIDEO_DIR missing before zip pipeline — expected: {FAKE_VIDEO_DIR}\n"
        f"Aborting — no zip or upload will occur."
    )

# POST-DOWNLOAD PIPELINE — Real - Fake.

for label in ['real', 'fake']:
    folder      = REAL_VIDEO_DIR if label == 'real' else FAKE_VIDEO_DIR
    total_label = total_real_secured if label == 'real' else total_fake_secured

    process_class_upload(
        label          = label,
        folder         = folder,
        zip_base       = LOCAL_ZIPS[label],
        manifest_path  = MANIFEST_PATHS[label],
        zip_s3_key     = S3_KEYS[f'{label}_zip'],
        manifest_s3_key= S3_KEYS[f'{label}_manifest'],
        video_counts   = video_counts,
        total_videos   = total_label,
    )

# FINAL CLEANUP

shutil.rmtree(TEMP_BASE, ignore_errors=True)
if os.path.exists(DOWNLOAD_SCRIPT):
    os.remove(DOWNLOAD_SCRIPT)
    print(f"\n[Cleanup] TUM download script wiped: {DOWNLOAD_SCRIPT}")

print("\n" + "═" * 60)
print("  FF++ Video Downloader — ALL DONE")
print(f"  Real zip : s3://{S3_BUCKET}/{S3_KEYS['real_zip']}")
print(f"  Fake zip : s3://{S3_BUCKET}/{S3_KEYS['fake_zip']}")
print("═" * 60 + "\n")


════════════════════════════════════════════════════════════
  FF++ Video Downloader — AWS SageMaker Production
  Target   : 5,000 videos  (1,000 real  +  4,000 fake across 4 subcategories)
  Bucket   : s3://deepfake-d-100k-dataset-tw26/datasets/FFPlus/
════════════════════════════════════════════════════════════

[Setup] Fetching TUM download script...
  Saved → /home/ec2-user/SageMaker/ffpp_download.py

[Download] Starting FF++ video download...

[Download] Pulling 1,000 videos — original...
By pressing any key to continue you confirm that you have agreed to the FaceForensics terms of use as described at:
http://kaldir.vc.in.tum.de/faceforensics/webpage/FaceForensics_TOS.pdf
***
Press any key to continue, or CTRL-C to exit.
Output path: /home/ec2-user/SageMaker/ffpp_temp/download_stage/real/original_sequences/youtube/c23/videos


100%|█████████▉| 999/1000 [26:20<00:01,  1.34s/it]

  Secured: 1,000 / 1,000 videos for original


100%|██████████| 1000/1000 [26:21<00:00,  1.58s/it]


  Moving original:   0%|          | 0/1000 [00:00<?, ?vid/s]

...

100%|█████████▉| 999/1000 [21:40<00:02,  2.11s/it]

  Secured: 1,000 / 1,000 videos for NeuralTextures


100%|██████████| 1000/1000 [21:43<00:00,  1.30s/it]


  Moving NeuralTextures:   0%|          | 0/1000 [00:00<?, ?vid/s]


[Download] Complete.
  Real  : 1,000 videos
  Fake  : 4,000 videos
  Total : 5,000 videos

────────────────────────────────────────────────────────────
Processing class: REAL
────────────────────────────────────────────────────────────

[1/5] Checking disk space...
  Folder size  : 1.80 GB
  Required free: 3.97 GB  (folder × 2.2)
  Actual free  : 457.63 GB

[2/5] Creating archive...
  Archiving real/  →  ffpp_real_videos.zip
  Archive ready: 1.80 GB

[3/5] Uploading archive to S3...
  Uploading ffpp_real_videos.zip  (1.80 GB)  →  s3://deepfake-d-100k-dataset-tw26/datasets/FFPlus/ffpp_real_videos.zip
  Verified: S3 size matches local (1.80 GB)

[4/5] Writing and uploading manifest...
FF++ Video Downloader — REAL Class Manifest
  Timestamp         : 2026-03-25 19:55:09 UTC
  Class             : real
  Total videos      : 1,000
  Archive size      : 1.80 GB
  S3 destination    : s3://deepfake-d-100k-dataset-tw26/datasets/FFPlus/ffpp_real_videos.zip
  Category breakdown:
  original  :  1,

3 Cell Modular Frame Extractor.

In [2]:
import os, shutil

BASE_DIR = '/home/ec2-user/SageMaker/extractor_temp'

if os.path.exists(BASE_DIR):
    shutil.rmtree(BASE_DIR, ignore_errors=True)
    print(f"Removed: {BASE_DIR}")
else:
    print("Nothing to clean.")

Removed: /home/ec2-user/SageMaker/extractor_temp


In [3]:
# Cell - 1.
# Delivery to Local EBS.
# Downloads ffpp_real_videos.zip and ffpp_fake_videos.zip from S3.
# Unpacks them to local SageMaker staging, and validates the unpacked structure.

import os
import shutil
import zipfile
import boto3
from datetime import datetime, timezone

# Configuration

S3_BUCKET  = 'deepfake-d-100k-dataset-tw26'
S3_PREFIX  = 'datasets/FFPlus'
BASE_DIR   = '/home/ec2-user/SageMaker/extractor_temp'

# Derived paths

RAW_ZIPS_DIR    = os.path.join(BASE_DIR, 'raw_zips')
RAW_VIDEOS_DIR  = os.path.join(BASE_DIR, 'raw_videos')
REAL_VIDEOS_DIR = os.path.join(RAW_VIDEOS_DIR, 'real')
FAKE_VIDEOS_DIR = os.path.join(RAW_VIDEOS_DIR, 'fake')

S3_OBJECTS = {
    'real' : f'{S3_PREFIX}/ffpp_real_videos.zip',
    'fake' : f'{S3_PREFIX}/ffpp_fake_videos.zip',
}

LOCAL_ZIPS = {
    'real' : os.path.join(RAW_ZIPS_DIR, 'ffpp_real_videos.zip'),
    'fake' : os.path.join(RAW_ZIPS_DIR, 'ffpp_fake_videos.zip'),
}

EXPECTED_FAKE_CATEGORIES = ['deepfakes', 'face2face', 'faceswap', 'neuraltextures']

# S3 client

s3 = boto3.client('s3')

# Helpers

import shutil

def make_dirs():
    # Clean old local staging from previous runs
    for d in [RAW_ZIPS_DIR, RAW_VIDEOS_DIR]:
        if os.path.exists(d):
            shutil.rmtree(d, ignore_errors=True)

    # Recreate clean directory structure
    for d in [RAW_ZIPS_DIR, RAW_VIDEOS_DIR, REAL_VIDEOS_DIR, FAKE_VIDEOS_DIR]:
        os.makedirs(d, exist_ok=True)

    print(f"  Clean staging directories ready under: {BASE_DIR}")


def download_zip(label):
    s3_key     = S3_OBJECTS[label]
    local_path = LOCAL_ZIPS[label]
    size_obj   = s3.head_object(Bucket=S3_BUCKET, Key=s3_key)['ContentLength']
    size_gb    = size_obj / (1024 ** 3)
    print(f"  Downloading {os.path.basename(s3_key)}  ({size_gb:.2f} GB)  ...")
    s3.download_file(S3_BUCKET, s3_key, local_path)
    local_size = os.path.getsize(local_path)
    if local_size != size_obj:
        raise RuntimeError(
            f"Download size mismatch for {label}.\n"
            f"  Expected : {size_obj} bytes\n"
            f"  Got      : {local_size} bytes"
        )
    print(f"  {label.upper()} zip saved → {local_path}  ({local_size / (1024**3):.2f} GB)")


def unpack_zip(label, extract_to):
    local_path = LOCAL_ZIPS[label]
    print(f"  Unpacking {os.path.basename(local_path)} → {extract_to} ...")
    with zipfile.ZipFile(local_path, 'r') as zf:
        zf.extractall(extract_to)
    print(f"  {label.upper()} zip unpacked.")


def validate_structure():
    """Confirm real dir and expected fake category subdirs exist after unpack."""
    errors = []

    if not os.path.isdir(REAL_VIDEOS_DIR):
        errors.append(f"Real video directory missing: {REAL_VIDEOS_DIR}")

    if not os.path.isdir(FAKE_VIDEOS_DIR):
        errors.append(f"Fake video directory missing: {FAKE_VIDEOS_DIR}")
    else:
        present = [d.lower() for d in os.listdir(FAKE_VIDEOS_DIR)
                   if os.path.isdir(os.path.join(FAKE_VIDEOS_DIR, d))]
        for cat in EXPECTED_FAKE_CATEGORIES:
            if cat not in present:
                errors.append(
                    f"Expected fake subcategory '{cat}' not found under {FAKE_VIDEOS_DIR}.\n"
                    f"  Found: {present}"
                )

    if errors:
        raise RuntimeError(
            "Unpacked structure validation failed:\n" +
            "\n".join(f"  - {e}" for e in errors)
        )

    print("  Structure validation PASSED.")
    print(f"    Real dir  : {REAL_VIDEOS_DIR}")
    print(f"    Fake dir  : {FAKE_VIDEOS_DIR}")
    for cat in EXPECTED_FAKE_CATEGORIES:
        cat_path = os.path.join(FAKE_VIDEOS_DIR, cat)
        mp4s = sum(
            len([f for f in files if f.endswith('.mp4')])
            for _, _, files in os.walk(cat_path)
        )
        print(f"      {cat:<20}: {mp4s:,} .mp4 files found")

# Main

print("\n" + "═" * 60)
print("  CELL 1 — THE DELIVERY TRUCK")
print(f"  {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")
print("═" * 60 + "\n")

print("[1/4] Creating local directories...")
make_dirs()

print("\n[2/4] Downloading from S3...")
for label in ['real', 'fake']:
    download_zip(label)

print("\n[3/4] Unpacking archives...")
# Unpack one level higher to prevent the "fake/fake/" Russian doll trap
unpack_zip('real', RAW_VIDEOS_DIR)
unpack_zip('fake', RAW_VIDEOS_DIR)

print("\n[4/4] Validating unpacked structure...")
validate_structure()

print("\n" + "─" * 60)
print("  CELL 1 COMPLETE")
print(f"  Zips downloaded to   : {RAW_ZIPS_DIR}")
print(f"  Videos extracted to  : {RAW_VIDEOS_DIR}")
print("─" * 60 + "\n")


════════════════════════════════════════════════════════════
  CELL 1 — THE DELIVERY TRUCK
  2026-03-26 16:56:29 UTC
════════════════════════════════════════════════════════════

[1/4] Creating local directories...
  Clean staging directories ready under: /home/ec2-user/SageMaker/extractor_temp

[2/4] Downloading from S3...
  REAL zip saved → /home/ec2-user/SageMaker/extractor_temp/raw_zips/ffpp_real_videos.zip  (1.80 GB)
  FAKE zip saved → /home/ec2-user/SageMaker/extractor_temp/raw_zips/ffpp_fake_videos.zip  (6.60 GB)

[3/4] Unpacking archives...
  Unpacking ffpp_real_videos.zip → /home/ec2-user/SageMaker/extractor_temp/raw_videos ...
  REAL zip unpacked.
  Unpacking ffpp_fake_videos.zip → /home/ec2-user/SageMaker/extractor_temp/raw_videos ...
  FAKE zip unpacked.

[4/4] Validating unpacked structure...
  Structure validation PASSED.
    Real dir  : /home/ec2-user/SageMaker/extractor_temp/raw_videos/real
    Fake dir  : /home/ec2-user/SageMaker/extractor_temp/raw_videos/fake
      d

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2 — THE SORTING HAT
# Scans unpacked videos, parses identities, builds the undirected identity graph,
# computes connected components, allocates components into 4 isolated master
# buckets, applies strict both-ID filtering, prints a full preflight audit,
# and saves extraction_map.json for Cell 3.
# Does NOT extract frames.
# ══════════════════════════════════════════════════════════════════════════════

import os
import re
import json
import glob
import heapq
from collections import defaultdict
from datetime import datetime, timezone

# ── Configuration — must match Cell 1 ────────────────────────────────────────

BASE_DIR        = '/home/ec2-user/SageMaker/extractor_temp'
RAW_VIDEOS_DIR  = os.path.join(BASE_DIR, 'raw_videos')
REAL_VIDEOS_DIR = os.path.join(RAW_VIDEOS_DIR, 'real')
FAKE_VIDEOS_DIR = os.path.join(RAW_VIDEOS_DIR, 'fake')

EXTRACTION_MAP_PATH = os.path.join(BASE_DIR, 'extraction_map.json')

# Manipulation categories and their assigned master bucket
CATEGORY_TO_BUCKET = {
    'deepfakes'      : 'A',
    'face2face'      : 'B',
    'faceswap'       : 'C',
    'neuraltextures' : 'D',
}
BUCKETS = ['A', 'B', 'C', 'D']

# ── Step A helpers — discover video files ────────────────────────────────────

def discover_real_videos():
    if not os.path.isdir(REAL_VIDEOS_DIR):
        raise RuntimeError(f"Real video directory missing: {REAL_VIDEOS_DIR}")
    paths = glob.glob(os.path.join(REAL_VIDEOS_DIR, '**', '*.mp4'), recursive=True)
    if not paths:
        raise RuntimeError(f"No .mp4 files found under {REAL_VIDEOS_DIR}")
    return sorted(paths)


def discover_fake_videos():
    if not os.path.isdir(FAKE_VIDEOS_DIR):
        raise RuntimeError(f"Fake video directory missing: {FAKE_VIDEOS_DIR}")

    category_paths = {}
    for cat in CATEGORY_TO_BUCKET:
        cat_dir = os.path.join(FAKE_VIDEOS_DIR, cat)
        if not os.path.isdir(cat_dir):
            raise RuntimeError(
                f"Expected fake category directory missing: {cat_dir}\n"
                f"  Available: {os.listdir(FAKE_VIDEOS_DIR)}"
            )
        paths = glob.glob(os.path.join(cat_dir, '**', '*.mp4'), recursive=True)
        if not paths:
            raise RuntimeError(f"No .mp4 files found under {cat_dir}")
        category_paths[cat] = sorted(paths)

    return category_paths

# ── Step B helpers — parse identities ────────────────────────────────────────

_REAL_PATTERN = re.compile(r'^(\d+)\.mp4$', re.IGNORECASE)
_FAKE_PATTERN = re.compile(r'^(\d+)_(\d+)\.mp4$', re.IGNORECASE)


def parse_real_id(filepath):
    """Return integer ID from a real filename like 000.mp4, or None on failure."""
    basename = os.path.basename(filepath)
    m = _REAL_PATTERN.match(basename)
    if not m:
        return None
    return int(m.group(1))


def parse_fake_ids(filepath):
    """Return (int, int) tuple of both IDs from a fake filename like 000_003.mp4, or None."""
    basename = os.path.basename(filepath)
    m = _FAKE_PATTERN.match(basename)
    if not m:
        return None
    return int(m.group(1)), int(m.group(2))

# ── Step C — build undirected identity graph ─────────────────────────────────

def build_identity_graph(fake_category_paths):
    """
    Nodes                 : all unique integer identity IDs seen in fake filenames.
    raw_pair_observations : count of every successfully parsed fake filename (includes duplicates across categories).
    unique_edges          : deduplicated set of undirected identity pairs — (min, max) normalized.
    Returns: (adjacency dict, raw_pair_observations, unique_edges, parse_failures dict, all_nodes set)
    """
    adjacency             = defaultdict(set)
    raw_pair_observations = 0
    unique_edges          = set()
    parse_failures        = defaultdict(list)

    for cat, paths in fake_category_paths.items():
        for fp in paths:
            ids = parse_fake_ids(fp)
            if ids is None:
                parse_failures[cat].append(fp)
                continue
            id_a, id_b = ids
            adjacency[id_a].add(id_b)
            adjacency[id_b].add(id_a)
            raw_pair_observations += 1
            unique_edges.add(tuple(sorted((id_a, id_b))))

    all_nodes = set(adjacency.keys())

    return dict(adjacency), raw_pair_observations, unique_edges, parse_failures, all_nodes

# ── Step D — compute connected components ────────────────────────────────────

def compute_connected_components(adjacency, all_nodes):
    """
    BFS over adjacency dict.
    Returns list of frozensets, each frozenset = one connected component.
    """
    visited    = set()
    components = []

    for start in sorted(all_nodes):
        if start in visited:
            continue
        component = set()
        queue     = [start]
        while queue:
            node = queue.pop()
            if node in visited:
                continue
            visited.add(node)
            component.add(node)
            for neighbour in adjacency.get(node, []):
                if neighbour not in visited:
                    queue.append(neighbour)
        components.append(frozenset(component))

    return components

# ── Step E — allocate components into 4 master buckets ───────────────────────

def allocate_components_to_buckets(components):
    """
    Greedy balanced allocation: sort components largest-first, assign each to
    the bucket with the fewest identities so far.
    Ties broken by bucket name order (A < B < C < D) for determinism.
    Returns: dict bucket_name -> set of identity IDs
    """
    heap = [(0, b) for b in BUCKETS]
    heapq.heapify(heap)

    bucket_identities = {b: set() for b in BUCKETS}

    sorted_components = sorted(components, key=lambda c: (-len(c), min(c)))

    for comp in sorted_components:
        size, bucket = heapq.heappop(heap)
        bucket_identities[bucket].update(comp)
        heapq.heappush(heap, (size + len(comp), bucket))

    return bucket_identities

# ── Step F — map videos into buckets with strict both-ID filter ───────────────

def map_real_videos_to_buckets(real_paths, bucket_identities):
    """
    Each original video NNN.mp4 goes to the bucket whose identity set contains NNN.
    Returns: dict bucket_name -> list of file paths, plus list of unmatched paths.
    """
    bucket_reals = {b: [] for b in BUCKETS}
    unmatched    = []

    for fp in real_paths:
        vid_id = parse_real_id(fp)
        if vid_id is None:
            unmatched.append(fp)
            continue
        placed = False
        for b in BUCKETS:
            if vid_id in bucket_identities[b]:
                bucket_reals[b].append(fp)
                placed = True
                break
        if not placed:
            unmatched.append(fp)

    return bucket_reals, unmatched


def map_fake_videos_strict(fake_category_paths, bucket_identities):
    """
    For each manipulation category, check BOTH IDs against the category's assigned bucket.
    A video is eligible only if BOTH IDs are in the assigned bucket's identity set.
    Returns:
      eligible : dict category -> list of eligible file paths
      skipped  : dict category -> list of skipped file paths
    """
    eligible = {}
    skipped  = {}

    for cat, bucket in CATEGORY_TO_BUCKET.items():
        id_set       = bucket_identities[bucket]
        cat_eligible = []
        cat_skipped  = []

        for fp in fake_category_paths.get(cat, []):
            ids = parse_fake_ids(fp)
            if ids is None:
                cat_skipped.append(fp)
                continue
            id_a, id_b = ids
            if id_a in id_set and id_b in id_set:
                cat_eligible.append(fp)
            else:
                cat_skipped.append(fp)

        eligible[cat] = cat_eligible
        skipped[cat]  = cat_skipped

    return eligible, skipped

# ── Step G — preflight audit ──────────────────────────────────────────────────

def print_audit(
    real_paths, fake_category_paths,
    all_nodes, raw_pair_observations, unique_edges, components,
    bucket_identities,
    bucket_reals, unmatched_reals,
    eligible_fakes, skipped_fakes,
    parse_failures,
):
    sep  = "═" * 60
    line = "─" * 60

    print(f"\n{sep}")
    print("  CELL 2 — PREFLIGHT AUDIT REPORT")
    print(f"  {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")
    print(sep)

    print(f"\n{line}")
    print("  [1] DISCOVERED VIDEOS")
    print(line)
    print(f"  Original (real) videos : {len(real_paths):>6,}")
    total_fake = sum(len(v) for v in fake_category_paths.values())
    print(f"  Total fake videos      : {total_fake:>6,}")
    for cat in CATEGORY_TO_BUCKET:
        print(f"    {cat:<20}: {len(fake_category_paths.get(cat, [])):>6,}")

    print(f"\n{line}")
    print("  [2] IDENTITY GRAPH")
    print(line)
    print(f"  Nodes (unique IDs)          : {len(all_nodes):>6,}")
    print(f"  Raw fake pair observations  : {raw_pair_observations:>6,}  (parsed fake files across all categories)")
    print(f"  Unique graph edges          : {len(unique_edges):>6,}  (deduplicated undirected identity pairs)")
    print(f"  Connected components        : {len(components):>6,}")

    comp_sizes = sorted([len(c) for c in components], reverse=True)
    if comp_sizes:
        print(f"  Largest component   : {comp_sizes[0]:>6,} identities")
        print(f"  Smallest component  : {comp_sizes[-1]:>6,} identities")
        print(f"  Median component    : {comp_sizes[len(comp_sizes)//2]:>6,} identities")
    else:
        print(f"  Largest component   :      0 identities (no components)")
        print(f"  Smallest component  :      0 identities")
        print(f"  Median component    :      0 identities")

    size_dist = defaultdict(int)
    for s in comp_sizes:
        size_dist[s] += 1
    print(f"  Component size distribution (size: count):")
    for size in sorted(size_dist.keys(), reverse=True)[:10]:
        print(f"    size {size:>4}: {size_dist[size]:>4} components")
    if len(size_dist) > 10:
        print(f"    ... ({len(size_dist) - 10} more size buckets)")

    print(f"\n{line}")
    print("  [3] MASTER BUCKET IDENTITY ALLOCATION")
    print(line)
    for b in BUCKETS:
        print(f"  Bucket {b} : {len(bucket_identities[b]):>6,} unique identities")

    print(f"\n{line}")
    print("  [4] REAL VIDEO MAPPING")
    print(line)
    for b in BUCKETS:
        print(f"  Bucket {b} real videos : {len(bucket_reals[b]):>6,}")
    if unmatched_reals:
        print(f"\n  WARNING: {len(unmatched_reals):,} real videos could not be placed in any bucket.")

    print(f"\n{line}")
    print("  [5] FAKE VIDEO ELIGIBILITY (strict both-ID filter)")
    print(line)
    for cat, bucket in CATEGORY_TO_BUCKET.items():
        n_elig = len(eligible_fakes.get(cat, []))
        n_skip = len(skipped_fakes.get(cat, []))
        n_tot  = len(fake_category_paths.get(cat, []))
        pct    = (n_elig / n_tot * 100) if n_tot else 0
        print(f"  {cat:<20} (Bucket {bucket}) : "
              f"{n_elig:>6,} eligible  /  {n_skip:>6,} skipped  "
              f"({pct:.1f}% pass rate)")

    total_failures = sum(len(v) for v in parse_failures.values())
    if total_failures:
        print(f"\n{line}")
        print("  [6] PARSE FAILURES (filenames that did not match expected pattern)")
        print(line)
        for cat, paths in parse_failures.items():
            if paths:
                print(f"  {cat}: {len(paths)} failures")
                for p in paths[:5]:
                    print(f"    {os.path.basename(p)}")
                if len(paths) > 5:
                    print(f"    ... ({len(paths) - 5} more)")

    LOW_THRESHOLD = 200
    print(f"\n{line}")
    print("  [7] WARNINGS")
    print(line)
    warned = False
    for cat in CATEGORY_TO_BUCKET:
        n = len(eligible_fakes.get(cat, []))
        if n < LOW_THRESHOLD:
            print(f"  WARNING: {cat} has only {n:,} eligible videos — below threshold of {LOW_THRESHOLD:,}.")
            warned = True
    for b in BUCKETS:
        n = len(bucket_reals[b])
        if n < LOW_THRESHOLD:
            print(f"  WARNING: Bucket {b} real videos only {n:,} — below threshold of {LOW_THRESHOLD:,}.")
            warned = True
    if not warned:
        print("  No warnings.")

    print(f"\n{sep}")
    print("  END OF AUDIT REPORT")
    print(sep + "\n")

# ── Step H — fail-hard checks ─────────────────────────────────────────────────

def fail_hard_checks(real_paths, fake_category_paths, all_nodes, eligible_fakes, bucket_reals):
    if not real_paths:
        raise RuntimeError("No original/real videos discovered. Aborting.")

    total_fake = sum(len(v) for v in fake_category_paths.values())
    if not total_fake:
        raise RuntimeError("No fake videos discovered. Aborting.")

    if not all_nodes:
        raise RuntimeError("Identity graph has zero nodes. Aborting.")

    for cat in CATEGORY_TO_BUCKET:
        if not fake_category_paths.get(cat):
            raise RuntimeError(
                f"Manipulation category '{cat}' has zero discovered videos. Aborting."
            )
        if not eligible_fakes.get(cat):
            raise RuntimeError(
                f"Manipulation category '{cat}' has zero eligible videos after "
                f"strict both-ID filtering. Aborting.\n"
                f"Check bucket allocation and identity graph."
            )

    for b in BUCKETS:
        if not bucket_reals.get(b):
            raise RuntimeError(
                f"Bucket {b} has zero eligible real videos after mapping. Aborting.\n"
                f"Check connected component allocation and identity graph."
            )

# ── Step I — save extraction map ─────────────────────────────────────────────

def save_extraction_map(
    bucket_identities,
    bucket_reals,
    eligible_fakes,
    components,
    real_paths,
    fake_category_paths,
    raw_pair_observations,
    unique_edges,
    skipped_fakes,
    unmatched_reals,
    parse_failures,
):
    component_membership = {
        str(i): sorted(list(comp))
        for i, comp in enumerate(
            sorted(components, key=lambda c: (-len(c), min(c)))
        )
    }

    extraction_map = {
        'generated_at'       : datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC'),
        'bucket_identities'  : {
            b: sorted(list(ids)) for b, ids in bucket_identities.items()
        },
        'real_video_paths'   : {
            b: sorted(bucket_reals[b]) for b in BUCKETS
        },
        'fake_video_paths'   : {
            cat: sorted(eligible_fakes.get(cat, []))
            for cat in CATEGORY_TO_BUCKET
        },
        'component_membership' : component_membership,
        'audit_stats'        : {
            'total_real_discovered'    : len(real_paths),
            'total_fake_discovered'    : {cat: len(v) for cat, v in fake_category_paths.items()},
            'total_real_eligible'      : {b: len(bucket_reals[b]) for b in BUCKETS},
            'total_fake_eligible'      : {cat: len(eligible_fakes.get(cat, [])) for cat in CATEGORY_TO_BUCKET},
            'total_fake_skipped'       : {cat: len(skipped_fakes.get(cat, [])) for cat in CATEGORY_TO_BUCKET},
            'total_real_unmatched'     : len(unmatched_reals),
            'graph_nodes'              : len(set().union(*[set(c) for c in components])) if components else 0,
            'raw_pair_observations'    : raw_pair_observations,
            'unique_graph_edges'       : len(unique_edges),
            'num_connected_components' : len(components),
            'largest_component_size'   : max(len(c) for c in components) if components else 0,
            'bucket_identity_counts'   : {b: len(ids) for b, ids in bucket_identities.items()},
            'parse_failures'           : {cat: len(v) for cat, v in parse_failures.items()},
        },
    }

    with open(EXTRACTION_MAP_PATH, 'w') as f:
        json.dump(extraction_map, f, indent=2)

    print(f"  Extraction map saved → {EXTRACTION_MAP_PATH}")
    size_kb = os.path.getsize(EXTRACTION_MAP_PATH) / 1024
    print(f"  File size: {size_kb:.1f} KB")

# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "═" * 60)
print("  CELL 2 — THE SORTING HAT")
print(f"  {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")
print("═" * 60 + "\n")

print("[A] Discovering video files...")
real_paths          = discover_real_videos()
fake_category_paths = discover_fake_videos()
print(f"  Real videos discovered : {len(real_paths):,}")
for cat, paths in fake_category_paths.items():
    print(f"  {cat:<20}: {len(paths):,} fake videos")

print("\n[B/C] Building identity graph from fake pairings...")
adjacency, raw_pair_observations, unique_edges, parse_failures, all_nodes = build_identity_graph(fake_category_paths)
print(f"  Graph nodes                : {len(all_nodes):,}")
print(f"  Raw fake pair observations : {raw_pair_observations:,}")
print(f"  Unique graph edges         : {len(unique_edges):,}")

print("\n[D] Computing connected components...")
components = compute_connected_components(adjacency, all_nodes)
print(f"  Connected components : {len(components):,}")
print(f"  Largest component   : {max((len(c) for c in components), default=0):,} identities")

print("\n[E] Allocating components into 4 master buckets...")
bucket_identities = allocate_components_to_buckets(components)
for b in BUCKETS:
    print(f"  Bucket {b} : {len(bucket_identities[b]):,} identities")

print("\n[F] Mapping videos into buckets with strict both-ID filtering...")
bucket_reals, unmatched_reals = map_real_videos_to_buckets(real_paths, bucket_identities)
eligible_fakes, skipped_fakes = map_fake_videos_strict(fake_category_paths, bucket_identities)
for b in BUCKETS:
    print(f"  Bucket {b} real videos : {len(bucket_reals[b]):,}")
for cat, bucket in CATEGORY_TO_BUCKET.items():
    print(f"  {cat:<20} (Bucket {bucket}) eligible : {len(eligible_fakes.get(cat, [])):,}")

print("\n[G] Running preflight audit...")
print_audit(
    real_paths, fake_category_paths,
    all_nodes, raw_pair_observations, unique_edges, components,
    bucket_identities,
    bucket_reals, unmatched_reals,
    eligible_fakes, skipped_fakes,
    parse_failures,
)

print("[H] Fail-hard validation checks...")
fail_hard_checks(real_paths, fake_category_paths, all_nodes, eligible_fakes, bucket_reals)
print("  All fail-hard checks passed.\n")

print("[I] Saving extraction map for Cell 3...")
save_extraction_map(
    bucket_identities,
    bucket_reals,
    eligible_fakes,
    components,
    real_paths,
    fake_category_paths,
    raw_pair_observations,
    unique_edges,
    skipped_fakes,
    unmatched_reals,
    parse_failures,
)

print("\n" + "─" * 60)
print("  CELL 2 COMPLETE — ready for Cell 3 (frame extraction)")
print(f"  Extraction map : {EXTRACTION_MAP_PATH}")
print("─" * 60 + "\n")


════════════════════════════════════════════════════════════
  CELL 2 — THE SORTING HAT
  2026-03-26 17:10:26 UTC
════════════════════════════════════════════════════════════

[A] Discovering video files...
  Real videos discovered : 1,000
  deepfakes           : 1,000 fake videos
  face2face           : 1,000 fake videos
  faceswap            : 1,000 fake videos
  neuraltextures      : 1,000 fake videos

[B/C] Building identity graph from fake pairings...
  Graph nodes                : 1,000
  Raw fake pair observations : 4,000
  Unique graph edges         : 500

[D] Computing connected components...
  Connected components : 500
  Largest component   : 2 identities

[E] Allocating components into 4 master buckets...
  Bucket A : 250 identities
  Bucket B : 250 identities
  Bucket C : 250 identities
  Bucket D : 250 identities

[F] Mapping videos into buckets with strict both-ID filtering...
  Bucket A real videos : 250
  Bucket B real videos : 250
  Bucket C real videos : 250
  Bucke

In [8]:
# CELL 3 — FF++ FRAME EXTRACTION ENGINE
# Final stage of the 3-cell modular pipeline:
#   Cell 1 = Download and unpack raw videos from S3.
#   Cell 2 = Build identity graph, allocate buckets, audit, save extraction_map.json.
#   Cell 3 = Load the map, extract frames, archive, upload to S3, clean up.
#
# Cell 3 is an EXECUTION ENGINE. It does NOT recompute graph logic, does NOT
# reassign buckets, and does NOT rebuild connected components. It trusts
# extraction_map.json produced by Cell 2 as the single source of truth.
#
# What it does:
#   1. Loads extraction_map.json
#   2. Validates that mapped video paths exist locally
#   3. Extracts 20 evenly-spaced raw frames per eligible video (OpenCV)
#   4. Writes frames into a provenance-preserving directory tree
#   5. Produces a human-readable manifest and a per-video CSV log
#   6. Archives the extracted frames into a single .zip
#   7. Uploads the archive + manifest + log to S3
#   8. Verifies every upload via head_object size comparison
#   9. Cleans local extracted frames and archive only after verified success

import os
import sys
import json
import csv
import cv2
import shutil
import zipfile
import boto3
import time
import numpy as np
from datetime import datetime, timezone

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

BASE_DIR = "/home/ec2-user/SageMaker/extractor_temp"
RAW_VIDEOS_DIR = os.path.join(BASE_DIR, "raw_videos")
EXTRACTION_MAP_PATH = os.path.join(BASE_DIR, "extraction_map.json")

EXTRACTED_DIR = os.path.join(BASE_DIR, "extracted_frames")
ARCHIVE_PATH = os.path.join(BASE_DIR, "ffpp_extracted_frames.zip")
MANIFEST_PATH = os.path.join(BASE_DIR, "ffpp_extracted_frames_manifest.txt")
LOG_CSV_PATH = os.path.join(BASE_DIR, "ffpp_frame_extraction_log.csv")

S3_BUCKET = "deepfake-d-100k-dataset-tw26"
S3_PREFIX = "datasets/FFPlus/processed"

TARGET_FRAMES_PER_VIDEO = 20  # buffered raw frame count before RetinaFace filtering

# Mapping from extraction_map.json keys -> local output subdirectories
REAL_BUCKET_MAP = {
    "bucket_a": "real/bucket_a",
    "bucket_b": "real/bucket_b",
    "bucket_c": "real/bucket_c",
    "bucket_d": "real/bucket_d",
}

FAKE_MANIP_MAP = {
    "deepfakes": "fake/deepfakes",
    "face2face": "fake/face2face",
    "faceswap": "fake/faceswap",
    "neuraltextures": "fake/neuraltextures",
}

# ─────────────────────────────────────────────────────────────────────────────
# HELPER: COMPUTE EVENLY-SPACED FRAME INDICES
# ─────────────────────────────────────────────────────────────────────────────

def compute_frame_indices(total_frames, target_count):
    """
    Return a list of unique, deterministic, evenly-spaced frame indices
    spread across the full video duration.

    If total_frames <= target_count, returns all available indices.
    """
    if total_frames <= 0:
        return []
    if total_frames <= target_count:
        return list(range(total_frames))
    # Evenly spaced across [0, total_frames - 1]
    indices = np.linspace(0, total_frames - 1, num=target_count, dtype=int)
    # Deduplicate while preserving order (unlikely with linspace but defensive)
    seen = set()
    unique = []
    for idx in indices:
        if idx not in seen:
            seen.add(idx)
            unique.append(int(idx))
    return unique


# ─────────────────────────────────────────────────────────────────────────────
# HELPER: EXTRACT FRAMES FROM A SINGLE VIDEO
# ─────────────────────────────────────────────────────────────────────────────

def extract_frames_from_video(video_path, output_dir, target_count, video_counter):
    """
    Opens a single video, extracts evenly-spaced frames, saves as JPG.

    Returns a dict with extraction stats for the CSV log.
    """
    result = {
        "video_path": video_path,
        "output_dir": output_dir,
        "total_frames_metadata": 0,
        "target_frames": target_count,
        "frames_saved": 0,
        "status": "failed",
        "failure_reason": "",
    }

    video_stem = os.path.splitext(os.path.basename(video_path))[0]

    if not os.path.isfile(video_path):
        result["failure_reason"] = "file_not_found"
        return result

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        result["failure_reason"] = "cv2_cannot_open"
        return result

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    result["total_frames_metadata"] = total_frames

    if total_frames <= 0:
        cap.release()
        result["failure_reason"] = "zero_or_negative_frame_count"
        return result

    indices = compute_frame_indices(total_frames, target_count)

    if not indices:
        cap.release()
        result["failure_reason"] = "no_valid_indices"
        return result

    os.makedirs(output_dir, exist_ok=True)
    saved = 0

    for frame_idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if not ret or frame is None:
            continue
        # Filename: {video_counter:04d}_{video_stem}_frame_{frame_idx:06d}.jpg
        fname = f"{video_counter:04d}_{video_stem}_frame_{frame_idx:06d}.jpg"
        out_path = os.path.join(output_dir, fname)
        try:
            cv2.imwrite(out_path, frame, [cv2.IMWRITE_JPEG_QUALITY, 95])
            saved += 1
        except Exception as e:
            # Log but don't crash
            print(f"  [WARN] Failed to write frame {frame_idx} for {video_stem}: {e}")

    cap.release()
    result["frames_saved"] = saved

    if saved == 0:
        result["status"] = "failed"
        result["failure_reason"] = "all_frame_reads_failed"
    elif saved < len(indices):
        result["status"] = "partial"
        result["failure_reason"] = f"only_{saved}_of_{len(indices)}_frames_read"
    else:
        result["status"] = "success"

    return result


# ─────────────────────────────────────────────────────────────────────────────
# HELPER: CREATE ZIP ARCHIVE
# ─────────────────────────────────────────────────────────────────────────────

def create_zip_archive(source_dir, archive_path):
    """
    Create a zip archive of the entire source directory tree.
    Returns the archive size in bytes, or raises on failure.
    """
    print(f"\n{'='*70}")
    print("CREATING ZIP ARCHIVE")
    print(f"{'='*70}")
    print(f"  Source : {source_dir}")
    print(f"  Target : {archive_path}")

    file_count = 0
    with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files in os.walk(source_dir):
            for f in files:
                abs_path = os.path.join(root, f)
                arc_name = os.path.relpath(abs_path, os.path.dirname(source_dir))
                zf.write(abs_path, arc_name)
                file_count += 1

    archive_size = os.path.getsize(archive_path)
    print(f"  Files archived : {file_count}")
    print(f"  Archive size   : {archive_size / (1024**2):.2f} MB")
    return archive_size


# ─────────────────────────────────────────────────────────────────────────────
# HELPER: S3 UPLOAD WITH VERIFICATION
# ─────────────────────────────────────────────────────────────────────────────

def upload_and_verify(s3_client, local_path, bucket, s3_key):
    """
    Upload a local file to S3, then verify via head_object size comparison.
    Raises RuntimeError if verification fails.
    """
    local_size = os.path.getsize(local_path)
    print(f"  Uploading: {os.path.basename(local_path)} ({local_size / (1024**2):.2f} MB)")
    print(f"    -> s3://{bucket}/{s3_key}")

    s3_client.upload_file(local_path, bucket, s3_key)

    # Verify
    head = s3_client.head_object(Bucket=bucket, Key=s3_key)
    remote_size = head['ContentLength']

    if remote_size != local_size:
        raise RuntimeError(
            f"UPLOAD VERIFICATION FAILED for {s3_key}: "
            f"local={local_size} bytes, remote={remote_size} bytes"
        )
    print(f"    Verified: {remote_size} bytes match.")


# ─────────────────────────────────────────────────────────────────────────────
# MAIN EXTRACTION PIPELINE
# ─────────────────────────────────────────────────────────────────────────────

start_time = time.time()
timestamp_str = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

print(f"{'='*70}")
print("CELL 3 — FF++ FRAME EXTRACTION ENGINE")
print(f"{'='*70}")
print(f"Timestamp       : {timestamp_str}")
print(f"Base dir        : {BASE_DIR}")
print(f"Extraction map  : {EXTRACTION_MAP_PATH}")
print(f"Frames/video    : {TARGET_FRAMES_PER_VIDEO}")
print()

# ─── STEP 1: LOAD EXTRACTION MAP ─────────────────────────────────────────────

if not os.path.isfile(EXTRACTION_MAP_PATH):
    raise FileNotFoundError(
        f"FATAL: extraction_map.json not found at {EXTRACTION_MAP_PATH}. "
        "Cell 2 must run first."
    )

with open(EXTRACTION_MAP_PATH, 'r') as f:
    extraction_map = json.load(f)

print(f"Loaded extraction_map.json ({os.path.getsize(EXTRACTION_MAP_PATH)} bytes)")

# ─── STEP 2: VALIDATE REQUIRED SECTIONS ──────────────────────────────────────

# We expect these top-level keys (adjust if Cell 2 uses different names)
# Attempt to locate real bucket paths and fake manipulation paths flexibly.

def get_nested(d, *keys):
    """Safely traverse nested dict keys."""
    current = d
    for k in keys:
        if isinstance(current, dict) and k in current:
            current = current[k]
        else:
            return None
    return current

# Try common structures Cell 2 might use
real_bucket_paths = {}
fake_manip_paths = {}

# Strategy: look for keys containing the bucket/manipulation video paths
# Common patterns: extraction_map["real_videos"]["bucket_a"], or
# extraction_map["eligible_real"]["bucket_a"], or
# extraction_map["buckets"]["bucket_a"]["real_videos"], etc.

# Real bucket key aliases: Cell 2 may use "A"/"B"/"C"/"D" instead of "bucket_a" etc.
REAL_BUCKET_ALIASES = {
    "bucket_a": ["bucket_a", "A", "a"],
    "bucket_b": ["bucket_b", "B", "b"],
    "bucket_c": ["bucket_c", "C", "c"],
    "bucket_d": ["bucket_d", "D", "d"],
}

for bkey in REAL_BUCKET_MAP:
    aliases = REAL_BUCKET_ALIASES[bkey]
    candidates = []
    for alias in aliases:
        candidates.extend([
            get_nested(extraction_map, "real_video_paths", alias),
            get_nested(extraction_map, "bucket_reals", alias),
            get_nested(extraction_map, "real_videos", alias),
            get_nested(extraction_map, "eligible_real", alias),
            get_nested(extraction_map, f"real_{alias}"),
            get_nested(extraction_map, "buckets", alias, "real_videos"),
            get_nested(extraction_map, "buckets", alias, "real"),
            get_nested(extraction_map, "buckets", alias, "eligible_real"),
        ])
    for c in candidates:
        if c is not None and isinstance(c, list):
            real_bucket_paths[bkey] = c
            break

for mkey in FAKE_MANIP_MAP:
    candidates = [
        get_nested(extraction_map, "fake_video_paths", mkey),
        get_nested(extraction_map, "eligible_fakes", mkey),
        get_nested(extraction_map, "fake_videos", mkey),
        get_nested(extraction_map, "eligible_fake", mkey),
        get_nested(extraction_map, f"fake_{mkey}"),
        get_nested(extraction_map, "manipulations", mkey, "eligible"),
        get_nested(extraction_map, "manipulations", mkey, "videos"),
        get_nested(extraction_map, "manipulations", mkey),
    ]
    for c in candidates:
        if c is not None and isinstance(c, list):
            fake_manip_paths[mkey] = c
            break

# If the flexible search didn't find everything, dump keys to help debug
if len(real_bucket_paths) < 4 or len(fake_manip_paths) < 4:
    print("\n[DEBUG] Top-level keys in extraction_map.json:")
    for k in extraction_map:
        v = extraction_map[k]
        vtype = type(v).__name__
        vlen = len(v) if isinstance(v, (list, dict)) else "n/a"
        print(f"  '{k}' -> {vtype} (len={vlen})")
        if isinstance(v, dict):
            for sk in v:
                sv = v[sk]
                svtype = type(sv).__name__
                svlen = len(sv) if isinstance(sv, (list, dict)) else "n/a"
                print(f"    '{sk}' -> {svtype} (len={svlen})")

    missing_real = [b for b in REAL_BUCKET_MAP if b not in real_bucket_paths]
    missing_fake = [m for m in FAKE_MANIP_MAP if m not in fake_manip_paths]

    raise ValueError(
        f"FATAL: Could not locate all required sections in extraction_map.json.\n"
        f"  Missing real buckets     : {missing_real}\n"
        f"  Missing fake manips      : {missing_fake}\n"
        f"  Adjust the key lookup patterns in Cell 3 to match Cell 2's output format."
    )

print("\nExtraction map sections located:")
for bkey, paths in real_bucket_paths.items():
    print(f"  Real {bkey:16s} : {len(paths)} videos")
for mkey, paths in fake_manip_paths.items():
    print(f"  Fake {mkey:16s} : {len(paths)} videos")

total_eligible = sum(len(v) for v in real_bucket_paths.values()) + \
                 sum(len(v) for v in fake_manip_paths.values())

if total_eligible == 0:
    raise ValueError("FATAL: No eligible videos found in extraction_map.json.")

print(f"\nTotal eligible videos: {total_eligible}")

# ─── STEP 3: VALIDATE LOCAL VIDEO PATHS EXIST ────────────────────────────────

print(f"\n{'='*70}")
print("VALIDATING LOCAL VIDEO PATHS")
print(f"{'='*70}")

missing_count = 0
present_count = 0

def validate_paths(path_list, label):
    global missing_count, present_count
    local_missing = 0
    for p in path_list:
        if os.path.isfile(p):
            present_count += 1
        else:
            missing_count += 1
            local_missing += 1
            if local_missing <= 3:
                print(f"  [MISSING] {label}: {p}")
    if local_missing > 3:
        print(f"  [MISSING] {label}: ... and {local_missing - 3} more")
    return local_missing

for bkey, paths in real_bucket_paths.items():
    validate_paths(paths, f"real/{bkey}")
for mkey, paths in fake_manip_paths.items():
    validate_paths(paths, f"fake/{mkey}")

print(f"\n  Videos present : {present_count}")
print(f"  Videos missing : {missing_count}")

if present_count == 0:
    raise FileNotFoundError(
        "FATAL: None of the mapped video paths exist locally. "
        "Did Cell 1 run? Is RAW_VIDEOS_DIR correct?"
    )

# ─── STEP 4: PREPARE OUTPUT DIRECTORY ────────────────────────────────────────

if os.path.exists(EXTRACTED_DIR):
    print(f"\nRemoving previous extracted_frames directory: {EXTRACTED_DIR}")
    shutil.rmtree(EXTRACTED_DIR)

os.makedirs(EXTRACTED_DIR, exist_ok=True)
print(f"Created output directory: {EXTRACTED_DIR}")

# ─── STEP 5: EXTRACT FRAMES ──────────────────────────────────────────────────

print(f"\n{'='*70}")
print("FRAME EXTRACTION")
print(f"{'='*70}")

csv_log_rows = []
video_counter = 0

# Counters for manifest
stats = {
    "real_attempted": {},
    "real_frames": {},
    "fake_attempted": {},
    "fake_frames": {},
    "videos_opened": 0,
    "videos_failed": 0,
    "total_frames": 0,
    "warnings": [],
}

def process_video_group(path_list, class_type, category_key, output_subdir):
    """
    Extract frames from all videos in a single category group.
    Updates stats and csv_log_rows in-place.
    """
    global video_counter

    out_dir = os.path.join(EXTRACTED_DIR, output_subdir)
    attempted = 0
    group_frames = 0

    for vpath in path_list:
        attempted += 1
        video_counter += 1

        result = extract_frames_from_video(
            video_path=vpath,
            output_dir=out_dir,
            target_count=TARGET_FRAMES_PER_VIDEO,
            video_counter=video_counter,
        )

        result["class_type"] = class_type
        result["category"] = category_key
        csv_log_rows.append(result)

        if result["status"] == "failed":
            stats["videos_failed"] += 1
            if result["failure_reason"] != "file_not_found":
                print(f"  [FAIL] {os.path.basename(vpath)}: {result['failure_reason']}")
        else:
            stats["videos_opened"] += 1
            if result["status"] == "partial":
                stats["warnings"].append(
                    f"Partial read: {os.path.basename(vpath)} "
                    f"({result['frames_saved']}/{TARGET_FRAMES_PER_VIDEO})"
                )

        group_frames += result["frames_saved"]
        stats["total_frames"] += result["frames_saved"]

        # Progress print every 50 videos
        if video_counter % 50 == 0:
            print(f"  ... processed {video_counter} videos so far ...")

    return attempted, group_frames


# ── Real buckets ──
print("\n--- Real Videos ---")
for bkey, subdir in REAL_BUCKET_MAP.items():
    paths = real_bucket_paths[bkey]
    print(f"\n  Processing real/{bkey} ({len(paths)} videos) ...")
    attempted, frames = process_video_group(paths, "real", bkey, subdir)
    stats["real_attempted"][bkey] = attempted
    stats["real_frames"][bkey] = frames
    print(f"    Attempted: {attempted} | Frames extracted: {frames}")

# ── Fake manipulations ──
print("\n--- Fake Videos ---")
for mkey, subdir in FAKE_MANIP_MAP.items():
    paths = fake_manip_paths[mkey]
    print(f"\n  Processing fake/{mkey} ({len(paths)} videos) ...")
    attempted, frames = process_video_group(paths, "fake", mkey, subdir)
    stats["fake_attempted"][mkey] = attempted
    stats["fake_frames"][mkey] = frames
    print(f"    Attempted: {attempted} | Frames extracted: {frames}")

print(f"\n{'='*70}")
print("EXTRACTION COMPLETE")
print(f"{'='*70}")
print(f"  Videos processed  : {video_counter}")
print(f"  Videos opened OK  : {stats['videos_opened']}")
print(f"  Videos failed     : {stats['videos_failed']}")
print(f"  Total frames      : {stats['total_frames']}")

if stats['total_frames'] == 0:
    raise RuntimeError("FATAL: Zero frames extracted. Something is critically wrong.")

# ─── STEP 6: WRITE CSV LOG ───────────────────────────────────────────────────

print(f"\nWriting CSV log: {LOG_CSV_PATH}")

csv_fields = [
    "video_path", "class_type", "category", "output_dir",
    "total_frames_metadata", "target_frames", "frames_saved",
    "status", "failure_reason",
]

with open(LOG_CSV_PATH, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=csv_fields, extrasaction='ignore')
    writer.writeheader()
    for row in csv_log_rows:
        writer.writerow(row)

print(f"  Rows written: {len(csv_log_rows)}")

# ─── STEP 7: CREATE ZIP ARCHIVE ──────────────────────────────────────────────

archive_size = create_zip_archive(EXTRACTED_DIR, ARCHIVE_PATH)

# ─── STEP 8: WRITE MANIFEST ──────────────────────────────────────────────────

print(f"\nWriting manifest: {MANIFEST_PATH}")

manifest_lines = []
manifest_lines.append("=" * 70)
manifest_lines.append("FF++ EXTRACTED FRAMES MANIFEST")
manifest_lines.append("=" * 70)
manifest_lines.append(f"Timestamp               : {timestamp_str}")
manifest_lines.append(f"Base directory           : {BASE_DIR}")
manifest_lines.append(f"Extraction map path      : {EXTRACTION_MAP_PATH}")
manifest_lines.append(f"Frames per video target  : {TARGET_FRAMES_PER_VIDEO}")
manifest_lines.append("")

manifest_lines.append("--- Real Videos by Bucket ---")
for bkey in REAL_BUCKET_MAP:
    attempted = stats["real_attempted"].get(bkey, 0)
    frames = stats["real_frames"].get(bkey, 0)
    manifest_lines.append(f"  {bkey:16s} : {attempted:4d} videos attempted, {frames:6d} frames extracted")

manifest_lines.append("")
manifest_lines.append("--- Fake Videos by Manipulation ---")
for mkey in FAKE_MANIP_MAP:
    attempted = stats["fake_attempted"].get(mkey, 0)
    frames = stats["fake_frames"].get(mkey, 0)
    manifest_lines.append(f"  {mkey:16s} : {attempted:4d} videos attempted, {frames:6d} frames extracted")

manifest_lines.append("")
manifest_lines.append("--- Summary ---")
manifest_lines.append(f"  Total videos processed   : {video_counter}")
manifest_lines.append(f"  Total videos opened OK   : {stats['videos_opened']}")
manifest_lines.append(f"  Total videos failed      : {stats['videos_failed']}")
manifest_lines.append(f"  Total frames extracted    : {stats['total_frames']}")
manifest_lines.append(f"  Archive size             : {archive_size / (1024**2):.2f} MB")

manifest_lines.append("")
manifest_lines.append("--- S3 Destinations ---")
manifest_lines.append(f"  Bucket  : {S3_BUCKET}")
manifest_lines.append(f"  Archive : s3://{S3_BUCKET}/{S3_PREFIX}/ffpp_extracted_frames.zip")
manifest_lines.append(f"  Manifest: s3://{S3_BUCKET}/{S3_PREFIX}/ffpp_extracted_frames_manifest.txt")
manifest_lines.append(f"  CSV Log : s3://{S3_BUCKET}/{S3_PREFIX}/ffpp_frame_extraction_log.csv")

if stats["warnings"]:
    manifest_lines.append("")
    manifest_lines.append(f"--- Warnings ({len(stats['warnings'])}) ---")
    for w in stats["warnings"][:50]:
        manifest_lines.append(f"  {w}")
    if len(stats["warnings"]) > 50:
        manifest_lines.append(f"  ... and {len(stats['warnings']) - 50} more warnings")

manifest_lines.append("")
elapsed = time.time() - start_time
manifest_lines.append(f"Extraction elapsed time: {elapsed:.1f} seconds")
manifest_lines.append("=" * 70)

with open(MANIFEST_PATH, 'w') as f:
    f.write("\n".join(manifest_lines))

print(f"  Manifest written ({len(manifest_lines)} lines)")

# ─── STEP 9: UPLOAD TO S3 ────────────────────────────────────────────────────

print(f"\n{'='*70}")
print("S3 UPLOAD")
print(f"{'='*70}")

s3 = boto3.client("s3")

uploads = [
    (ARCHIVE_PATH, f"{S3_PREFIX}/ffpp_extracted_frames.zip"),
    (MANIFEST_PATH, f"{S3_PREFIX}/ffpp_extracted_frames_manifest.txt"),
    (LOG_CSV_PATH, f"{S3_PREFIX}/ffpp_frame_extraction_log.csv"),
]

for local_path, s3_key in uploads:
    upload_and_verify(s3, local_path, S3_BUCKET, s3_key)

print("\nAll uploads verified successfully.")

# ─── STEP 10: LOCAL CLEANUP ──────────────────────────────────────────────────

print(f"\n{'='*70}")
print("LOCAL CLEANUP")
print(f"{'='*70}")

# Clean extracted frames directory (the archive is already on S3)
if os.path.exists(EXTRACTED_DIR):
    shutil.rmtree(EXTRACTED_DIR)
    print(f"  Removed: {EXTRACTED_DIR}")

# Clean local archive (already uploaded and verified)
if os.path.isfile(ARCHIVE_PATH):
    os.remove(ARCHIVE_PATH)
    print(f"  Removed: {ARCHIVE_PATH}")

# Keep manifest and CSV log locally for quick reference
print(f"  Kept locally: {MANIFEST_PATH}")
print(f"  Kept locally: {LOG_CSV_PATH}")

# NOTE: Raw videos from Cell 1 are NOT deleted here.
# They remain in {RAW_VIDEOS_DIR} in case you need to re-run extraction.
# Delete them manually or in a separate cleanup cell if disk space is needed.
print(f"  Raw videos preserved: {RAW_VIDEOS_DIR}")

# ─── DONE ─────────────────────────────────────────────────────────────────────

total_elapsed = time.time() - start_time
print(f"\n{'='*70}")
print("CELL 3 COMPLETE")
print(f"{'='*70}")
print(f"  Total time     : {total_elapsed:.1f} seconds ({total_elapsed/60:.1f} min)")
print(f"  Frames on S3   : s3://{S3_BUCKET}/{S3_PREFIX}/ffpp_extracted_frames.zip")
print(f"  Manifest on S3 : s3://{S3_BUCKET}/{S3_PREFIX}/ffpp_extracted_frames_manifest.txt")
print(f"  Log on S3      : s3://{S3_BUCKET}/{S3_PREFIX}/ffpp_frame_extraction_log.csv")
print(f"  Total frames   : {stats['total_frames']}")
print(f"  Failed videos  : {stats['videos_failed']}")
print()

CELL 3 — FF++ FRAME EXTRACTION ENGINE
Timestamp       : 2026-03-26 17:33:35 UTC
Base dir        : /home/ec2-user/SageMaker/extractor_temp
Extraction map  : /home/ec2-user/SageMaker/extractor_temp/extraction_map.json
Frames/video    : 20

Loaded extraction_map.json (193461 bytes)

Extraction map sections located:
  Real bucket_a         : 250 videos
  Real bucket_b         : 250 videos
  Real bucket_c         : 250 videos
  Real bucket_d         : 250 videos
  Fake deepfakes        : 250 videos
  Fake face2face        : 250 videos
  Fake faceswap         : 250 videos
  Fake neuraltextures   : 250 videos

Total eligible videos: 2000

VALIDATING LOCAL VIDEO PATHS

  Videos present : 2000
  Videos missing : 0
Created output directory: /home/ec2-user/SageMaker/extractor_temp/extracted_frames

FRAME EXTRACTION

--- Real Videos ---

  Processing real/bucket_a (250 videos) ...
  ... processed 50 videos so far ...
  ... processed 100 videos so far ...
  ... processed 150 videos so far ...
  ...

CredentialRetrievalError: Error when retrieving credentials from iam-role: Credential refresh failed, response did not contain: access_key, secret_key, token, expiry_time

Manual Upload from EBS to S3 Due to AWS Credential Loss.

In [3]:
import os
import boto3

S3_BUCKET = "deepfake-d-100k-dataset-tw26"
S3_PREFIX = "datasets/FFPlus/processed"
BASE_DIR  = "/home/ec2-user/SageMaker/extractor_temp"

ARCHIVE_PATH  = os.path.join(BASE_DIR, "ffpp_extracted_frames.zip")
MANIFEST_PATH = os.path.join(BASE_DIR, "ffpp_extracted_frames_manifest.txt")
LOG_CSV_PATH  = os.path.join(BASE_DIR, "ffpp_frame_extraction_log.csv")

files_to_upload = [
    (ARCHIVE_PATH,  f"{S3_PREFIX}/ffpp_extracted_frames.zip"),
    (MANIFEST_PATH, f"{S3_PREFIX}/ffpp_extracted_frames_manifest.txt"),
    (LOG_CSV_PATH,  f"{S3_PREFIX}/ffpp_frame_extraction_log.csv"),
]

def human_mb(n):
    return f"{n / (1024**2):.2f} MB"

s3 = boto3.client("s3")

print("=== S3 UPLOAD ===\n")

for local_path, s3_key in files_to_upload:
    if not os.path.exists(local_path):
        raise FileNotFoundError(f"Missing local file: {local_path}")

    local_size = os.path.getsize(local_path)
    if local_size == 0:
        raise RuntimeError(f"Local file is empty: {local_path}")

    print(f"Uploading: {os.path.basename(local_path)} ({human_mb(local_size)})")
    print(f"  -> s3://{S3_BUCKET}/{s3_key}")

    s3.upload_file(local_path, S3_BUCKET, s3_key)

    head = s3.head_object(Bucket=S3_BUCKET, Key=s3_key)
    remote_size = head["ContentLength"]

    if remote_size != local_size:
        raise RuntimeError(
            f"Size mismatch after upload for {s3_key}\n"
            f"  Local  : {local_size:,} bytes\n"
            f"  Remote : {remote_size:,} bytes"
        )

    print("  Verified.\n")

print("ALL FILES UPLOADED AND VERIFIED.")

=== S3 UPLOAD ===

Uploading: ffpp_extracted_frames.zip (5223.55 MB)
  -> s3://deepfake-d-100k-dataset-tw26/datasets/FFPlus/processed/ffpp_extracted_frames.zip
  Verified.

Uploading: ffpp_extracted_frames_manifest.txt (0.00 MB)
  -> s3://deepfake-d-100k-dataset-tw26/datasets/FFPlus/processed/ffpp_extracted_frames_manifest.txt
  Verified.

Uploading: ffpp_frame_extraction_log.csv (0.34 MB)
  -> s3://deepfake-d-100k-dataset-tw26/datasets/FFPlus/processed/ffpp_frame_extraction_log.csv
  Verified.

ALL FILES UPLOADED AND VERIFIED.


RetinaFace Face Extraction on Processed FF++ Data.

- Installers.

In [4]:
!pip install retina-face imagehash pybktree opencv-python-headless

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 164.8 MB/s  0:00:00 eta 0:00:01
Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.8 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.2/572.2 MB 58.9 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 203.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 166.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 121.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 245.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 141.8 MB/s  0:00:00
  Created wheel for pybktree: filename=pybktree-1.1-py3-none-any.whl size=5025 sha256=84fcc401fa882e067fd99e36e

In [2]:
%pip install -U tf-keras

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 60.7 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [1]:
# FF++ RetinaFace Face Extractor — AWS SageMaker Production.

# Architecture:
#   1.  Wipe and recreate local staging directories (Fix 1)
#   2.  Download ffpp_extracted_frames.zip from S3
#   3.  Unpack via native subprocess unzip (Fix 4)
#   4.  Validate unpacked folder structure (Fix 3)
#   5.  Scan all frame paths per category
#   6.  Build FF++ identity graph from fake stems → connected components (Fix 6)
#   7.  Assign whole connected components to train/val/test splits (Fix 6/7)
#       - component membership enforces zero leakage of target/donor identities
#       - real stems resolved to splits via their component membership
#   8.  Run RetinaFace / blur / pHash dedup / resize (260×260) per frame
#       - per quota-bucket BK trees preserve dedup isolation (Fix 8)
#       - pbar.update(1) called on every accepted face (Fix 5)
#   9.  Fail hard if any quota bucket is underfilled (Fix 2)
#  10.  Zip final accepted faces
#  11.  Write CSV log + manifest (includes graph/component stats) (Fix 10)
#  12.  Upload zip + manifest + CSV to S3
#  13.  Verify uploads (head_object size check)
#  14.  Clean local staging only after verification

import csv
import glob
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"     # Bypasses the Keras 3 architecture crash.
os.environ['TF_CUDNN_USE_AUTOTUNE'] = '0'
import random
import re
import shutil
import subprocess
import time
import warnings
from collections import defaultdict
from datetime import datetime, timezone

import boto3
import cv2
import imagehash
import numpy as np
import pybktree
from PIL import Image
from retinaface import RetinaFace
from tqdm import tqdm

warnings.filterwarnings("ignore")

# Determinism
random.seed(42)
np.random.seed(42)

# CONFIGURATION

S3_BUCKET        = 'deepfake-d-100k-dataset-tw26'
S3_INPUT_KEY     = 'datasets/FFPlus/processed/ffpp_extracted_frames.zip'
S3_OUTPUT_PREFIX = 'datasets/FFPlus/final'

BASE_DIR         = '/home/ec2-user/SageMaker/ffpp_face_stage'
RAW_ZIP_DIR      = os.path.join(BASE_DIR, 'raw_zip')
RAW_FRAMES_DIR   = os.path.join(BASE_DIR, 'raw_frames')
FINAL_FACES_DIR  = os.path.join(BASE_DIR, 'final_faces')
LOGS_DIR         = os.path.join(BASE_DIR, 'logs')

LOCAL_ZIP_IN     = os.path.join(RAW_ZIP_DIR, 'ffpp_extracted_frames.zip')
FINAL_ZIP_BASE   = os.path.join(BASE_DIR,    'ffpp_final_faces_20k')   # .zip appended by make_archive
CSV_LOG_PATH     = os.path.join(LOGS_DIR,    'ffpp_final_faces_20k_log.csv')
MANIFEST_PATH    = os.path.join(LOGS_DIR,    'ffpp_final_faces_20k_manifest.txt')

S3_OUT_KEYS = {
    'zip'      : f'{S3_OUTPUT_PREFIX}/ffpp_final_faces_20k.zip',
    'manifest' : f'{S3_OUTPUT_PREFIX}/ffpp_final_faces_20k_manifest.txt',
    'csv'      : f'{S3_OUTPUT_PREFIX}/ffpp_final_faces_20k_log.csv',
}

# Validated quality thresholds.
CONFIDENCE_THRESHOLD = 0.90
MIN_FACE_SIZE        = 30
MIN_FACE_RATIO       = 0.005
BLUR_THRESHOLD       = 25
PHASH_HAMMING_MAX    = 2
PADDING              = 25
RESIZE_TARGET        = (260, 260)   # EfficientNet-B2 native resolution

# Production quotas — 20K total (10K real + 10K fake)
target_quotas = {
    'train_real'               : 7000,
    'val_real'                 : 1500,
    'test_real'                : 1500,
    'train_fake_deepfakes'     : 1750,
    'val_fake_deepfakes'       : 375,
    'test_fake_deepfakes'      : 375,
    'train_fake_face2face'     : 1750,
    'val_fake_face2face'       : 375,
    'test_fake_face2face'      : 375,
    'train_fake_faceswap'      : 1750,
    'val_fake_faceswap'        : 375,
    'test_fake_faceswap'       : 375,
    'train_fake_neuraltextures': 1750,
    'val_fake_neuraltextures'  : 375,
    'test_fake_neuraltextures' : 375,
}

SPLIT_ORDER = ['train', 'val', 'test']

# Maps input folder name → quota category suffix
INPUT_CATEGORY_MAP = {
    'real'           : 'real',
    'deepfakes'      : 'fake_deepfakes',
    'face2face'      : 'fake_face2face',
    'faceswap'       : 'fake_faceswap',
    'neuraltextures' : 'fake_neuraltextures',
}

FAKE_CATS = ['deepfakes', 'face2face', 'faceswap', 'neuraltextures']

# S3 client
s3 = boto3.client('s3')

# HELPERS — FILE SYSTEM

def clean_and_create_dirs():
    """Fix 1: Wipe and recreate all staging dirs to prevent cross-run contamination."""
    for d in [RAW_ZIP_DIR, RAW_FRAMES_DIR, FINAL_FACES_DIR, LOGS_DIR]:
        shutil.rmtree(d, ignore_errors=True)
        os.makedirs(d, exist_ok=True)
    print(f"  Staging directories wiped and recreated under: {BASE_DIR}")


def make_output_split_dirs():
    for split in SPLIT_ORDER:
        for cat_suffix in INPUT_CATEGORY_MAP.values():
            os.makedirs(os.path.join(FINAL_FACES_DIR, split, cat_suffix), exist_ok=True)

# HELPERS — S3

def download_from_s3(s3_key, local_path):
    size_obj = s3.head_object(Bucket=S3_BUCKET, Key=s3_key)['ContentLength']
    size_gb  = size_obj / (1024 ** 3)
    print(f"  Downloading s3://{S3_BUCKET}/{s3_key}  ({size_gb:.2f} GB) ...")
    s3.download_file(S3_BUCKET, s3_key, local_path)
    local_size = os.path.getsize(local_path)
    if local_size != size_obj:
        raise RuntimeError(
            f"Download size mismatch.\n  Expected: {size_obj}\n  Got: {local_size}"
        )
    print(f"  Saved → {local_path}  ({local_size / (1024**3):.2f} GB)")


def upload_to_s3(local_path, s3_key):
    size_gb = os.path.getsize(local_path) / (1024 ** 3)
    print(f"  Uploading {os.path.basename(local_path)}  ({size_gb:.2f} GB)  → s3://{S3_BUCKET}/{s3_key}")
    s3.upload_file(local_path, S3_BUCKET, s3_key)


def verify_s3_upload(local_path, s3_key):
    local_size = os.path.getsize(local_path)
    try:
        s3_size = s3.head_object(Bucket=S3_BUCKET, Key=s3_key)['ContentLength']
    except Exception as e:
        raise RuntimeError(f"S3 verification failed — head_object error: {e}")
    if s3_size != local_size:
        raise RuntimeError(
            f"S3 size mismatch for {s3_key}.\n"
            f"  Local: {local_size} bytes\n"
            f"  S3   : {s3_size} bytes\n"
            f"Upload may be incomplete. Local artifacts preserved."
        )
    print(f"  Verified: {os.path.basename(local_path)}  ({s3_size / (1024**3):.2f} GB)")

# HELPERS — FILENAME PARSING

_STEM_RE = re.compile(r'^(.+?)_frame_\d+')

def parse_source_video_stem(filename):
    """
    Strip '_frame_XXXXXX.jpg' suffix to recover the source video stem.
      '1001_000_frame_000123.jpg'      -> '1001_000'
      '1001_000_003_frame_000100.jpg'  -> '1001_000_003'
    Falls back to bare basename (no extension) if marker is absent.
    """
    basename = os.path.splitext(os.path.basename(filename))[0]
    m = _STEM_RE.match(basename)
    return m.group(1) if m else basename

def parse_fake_stem_ids(stem):
    """
    Parse both integer IDs from a fake source stem, ignoring junk prefixes.
    e.g., '1001_000_003' -> (0, 3)
    """
    parts = stem.split('_')
    if len(parts) >= 2:
        try:
            return int(parts[-2]), int(parts[-1])
        except ValueError:
            return None
    return None

def parse_real_stem_id(stem):
    """
    Parse the single integer ID from a real source stem, ignoring junk prefixes.
    e.g., '1001_000' -> 0
    """
    parts = stem.split('_')
    if len(parts) >= 1:
        try:
            return int(parts[-1])
        except ValueError:
            return None
    return None

# FIX 6 — FF++ IDENTITY GRAPH + CONNECTED COMPONENTS + SPLIT ASSIGNMENT

def build_identity_graph(fake_stems):
    """
    Build undirected graph over all fake source-video stems.
    '000_003' adds an undirected edge between node 0 and node 3.
    Returns: adjacency dict, set of all nodes, set of unique normalized edges.
    """
    adjacency    = defaultdict(set)
    unique_edges = set()

    for stem in fake_stems:
        ids = parse_fake_stem_ids(stem)
        if ids is None:
            continue
        id_a, id_b = ids
        adjacency[id_a].add(id_b)
        adjacency[id_b].add(id_a)
        unique_edges.add(tuple(sorted((id_a, id_b))))

    return dict(adjacency), set(adjacency.keys()), unique_edges


def compute_connected_components(adjacency, all_nodes):
    """BFS connected components. Returns list of frozensets."""
    visited    = set()
    components = []
    for start in sorted(all_nodes):
        if start in visited:
            continue
        component = set()
        queue     = [start]
        while queue:
            node = queue.pop()
            if node in visited:
                continue
            visited.add(node)
            component.add(node)
            for nb in adjacency.get(node, []):
                if nb not in visited:
                    queue.append(nb)
        components.append(frozenset(component))
    return components


def assign_components_to_splits(components):
    """
    Assign whole connected components to train/val/test — never split a component.

    Strategy:
    - Shuffle components with fixed seed, then sort largest-first (stable tie-break)
    - Fill train up to 70% of total identities, then val up to 15%, test gets the rest
    - Returns: node_id -> split name, and identity counts per split
    """
    comp_list = list(components)
    random.shuffle(comp_list)                         # seed=42 set globally
    comp_list.sort(key=lambda c: -len(c))             # stable: largest first

    total_ids    = sum(len(c) for c in comp_list)
    train_target = int(total_ids * 0.70)
    val_target   = int(total_ids * 0.15)

    split_counts  = {'train': 0, 'val': 0, 'test': 0}
    node_to_split = {}

    for comp in comp_list:
        if split_counts['train'] < train_target:
            chosen = 'train'
        elif split_counts['val'] < val_target:
            chosen = 'val'
        else:
            chosen = 'test'
        split_counts[chosen] += len(comp)
        for node in comp:
            node_to_split[node] = chosen

    return node_to_split, split_counts


def resolve_stem_split(stem, node_to_split, is_fake):
    """
    Resolve split assignment for a source-video stem via component membership.

    Real  '000'     -> look up node 0
    Fake  '000_003' -> both IDs must map to the same split (component guarantee)
    Returns split string or None if stem cannot be mapped.
    """
    if is_fake:
        ids = parse_fake_stem_ids(stem)
        if ids is None:
            return None
        id_a, id_b = ids
        split_a = node_to_split.get(id_a)
        split_b = node_to_split.get(id_b)
        # Component guarantee: these should always agree; guard defensively
        if split_a is None or split_b is None or split_a != split_b:
            return None
        return split_a
    else:
        node_id = parse_real_stem_id(stem)
        if node_id is None:
            return None
        return node_to_split.get(node_id)

# HELPERS — RETINAFACE CROP ENGINE

def master_crop_engine(
    img_path,
    save_path,
    bucket_tree,
    padding=PADDING,
    min_face_size=MIN_FACE_SIZE,
    min_face_ratio=MIN_FACE_RATIO,
    blur_threshold=BLUR_THRESHOLD,
    confidence_threshold=CONFIDENCE_THRESHOLD,
):
    """
    Load -> DOWNSCALE FOR DETECTION -> detect -> select largest -> 
    UPSCALE BOX -> quality gate -> crop+pad -> blur check -> resize -> pHash -> save.
    """
    try:
        img_cv = cv2.imread(img_path)
        if img_cv is None:
            return False, "Corrupted Image", 0.0, 0.0, "", "[]"

        height, width = img_cv.shape[:2]

        # --- THE 1080p BYPASS ---
        # Downscale by 50% strictly for the RetinaFace scan to prevent VRAM deadlock
        scale_factor = 0.5
        small_cv = cv2.resize(img_cv, (0, 0), fx=scale_factor, fy=scale_factor)

        faces = RetinaFace.detect_faces(small_cv)
        if not isinstance(faces, dict) or len(faces) == 0:
            return False, "No Face Detected", 0.0, 0.0, "", "[]"

        largest_area = 0
        best_face    = None
        for face in faces.values():
            box = face.get('facial_area')
            if box is None:
                continue
            area = (box[2] - box[0]) * (box[3] - box[1])
            if area > largest_area:
                largest_area = area
                best_face    = face

        if best_face is None:
            return False, "No Valid Face Found", 0.0, 0.0, "", "[]"

        # --- UPSCALE THE BOX ---
        # Multiply the coordinates by 2 so we crop from the original 1080p High-Res image
        raw_box = best_face['facial_area']
        box = [int(coord / scale_factor) for coord in raw_box]
        
        str_box    = f"[{box[0]}, {box[1]}, {box[2]}, {box[3]}]"
        confidence = best_face['score']

        if confidence < confidence_threshold:
            return False, "Low Confidence", confidence, 0.0, "", str_box

        face_w = box[2] - box[0]
        face_h = box[3] - box[1]

        if face_w < min_face_size or face_h < min_face_size:
            return False, "Resolution Too Small", confidence, 0.0, "", str_box

        if (face_w * face_h) / (width * height) < min_face_ratio:
            return False, "Face Ratio Too Small", confidence, 0.0, "", str_box

        x_min = max(0,      int(box[0]) - padding)
        y_min = max(0,      int(box[1]) - padding)
        x_max = min(width,  int(box[2]) + padding)
        y_max = min(height, int(box[3]) + padding)

        cropped_cv  = img_cv[y_min:y_max, x_min:x_max]
        gray_crop   = cv2.cvtColor(cropped_cv, cv2.COLOR_BGR2GRAY)
        blur_val    = cv2.Laplacian(gray_crop, cv2.CV_64F).var()

        if blur_val < blur_threshold:
            return False, "Motion Blur Detected", confidence, blur_val, "", str_box

        cropped_pil  = Image.fromarray(cv2.cvtColor(cropped_cv, cv2.COLOR_BGR2RGB))
        standardized = cropped_pil.resize(RESIZE_TARGET, Image.BICUBIC)

        new_hash = imagehash.phash(standardized)
        matches  = bucket_tree.find(new_hash, PHASH_HAMMING_MAX)
        if matches:
            return False, f"Duplicate Face (Hamming <= {PHASH_HAMMING_MAX})", confidence, blur_val, str(new_hash), str_box

        bucket_tree.add(new_hash)
        standardized.save(save_path, format='JPEG', quality=95)

        return True, "Accepted", confidence, blur_val, str(new_hash), str_box

    except Exception as exc:
        return False, f"Engine Error: {str(exc)}", 0.0, 0.0, "", "[]"

# HELPERS — QUOTA UTILITIES

def quota_key(split, cat_suffix):
    return f"{split}_{cat_suffix}"


def quotas_all_filled(accepted_counts):
    return all(accepted_counts[k] >= target_quotas[k] for k in target_quotas)

# CORE PROCESSING

def process_category(
    cat_key,
    cat_suffix,
    frame_paths,
    node_to_split,
    is_fake,
    accepted_counts,
    rejection_reasons,
    csv_writer,
    bk_trees,
    pbar,
):
    """
    Group frames by source-video stem.
    Resolve each stem's split via component membership (Fix 6).
    Process frames through crop engine; accept into the correct split quota bucket.
    pbar.update(1) fired on each accepted face (Fix 5).
    """
    groups = defaultdict(list)
    for fp in frame_paths:
        stem = parse_source_video_stem(fp)
        groups[stem].append(fp)

    group_keys = sorted(groups.keys())
    random.shuffle(group_keys)   # seed=42 set globally

    for stem in group_keys:
        if quotas_all_filled(accepted_counts):
            return

        assigned_split = resolve_stem_split(stem, node_to_split, is_fake)
        if assigned_split is None:
            for fp in groups[stem]:
                rejection_reasons['Unmapped Stem'] += 1
                csv_writer.writerow([
                    fp, stem, 'none', cat_suffix,
                    'Rejected', 'Unmapped Stem',
                    0.0, 0.0, '', '[]'
                ])
            continue

        quota_k = quota_key(assigned_split, cat_suffix)

        # Skip group entirely if its designated quota bucket is already full
        if accepted_counts[quota_k] >= target_quotas[quota_k]:
            continue

        tree    = bk_trees[quota_k]
        out_dir = os.path.join(FINAL_FACES_DIR, assigned_split, cat_suffix)

        for img_path in groups[stem]:
            if accepted_counts[quota_k] >= target_quotas[quota_k]:
                break

            file_name = os.path.basename(img_path)
            save_path = os.path.join(out_dir, file_name)

            passed, reason, conf, blur, phash_val, bbox = master_crop_engine(
                img_path, save_path, tree
            )

            if passed:
                accepted_counts[quota_k] += 1
                pbar.update(1)   # Fix 5: live update on every acceptance
                csv_writer.writerow([
                    img_path, stem, assigned_split, cat_suffix,
                    'Accepted', 'None',
                    round(conf, 4), round(blur, 2), phash_val, bbox
                ])
            else:
                rejection_reasons[reason] += 1
                csv_writer.writerow([
                    img_path, stem, assigned_split, cat_suffix,
                    'Rejected', reason,
                    round(conf, 4), round(blur, 2), phash_val, bbox
                ])

# MANIFEST

def write_manifest(
    accepted_counts,
    rejection_reasons,
    zip_path,
    start_time,
    graph_stats,
    split_identity_counts,
):
    timestamp = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')
    elapsed   = time.time() - start_time
    zip_size  = os.path.getsize(zip_path) / (1024 ** 3) if os.path.exists(zip_path) else 0.0
    total_acc = sum(accepted_counts.values())
    total_rej = sum(rejection_reasons.values())

    lines = [
        "FF++ RetinaFace Face Extractor — Production Manifest",
        "=" * 60,
        f"  Timestamp              : {timestamp}",
        f"  Elapsed                : {elapsed:.0f} seconds  ({elapsed/60:.1f} min)",
        f"  Input S3 object        : s3://{S3_BUCKET}/{S3_INPUT_KEY}",
        f"  Local workspace        : {BASE_DIR}",
        "",
        "  Quality thresholds (validated sandbox defaults):",
        f"    Confidence           : >= {CONFIDENCE_THRESHOLD}",
        f"    Min face size        : {MIN_FACE_SIZE} px",
        f"    Min face ratio       : {MIN_FACE_RATIO}",
        f"    Blur threshold       : {BLUR_THRESHOLD} (Laplacian variance)",
        f"    pHash Hamming max    : <= {PHASH_HAMMING_MAX}",
        f"    Padding              : {PADDING} px",
        f"    Resize target        : {RESIZE_TARGET[0]}x{RESIZE_TARGET[1]}",
        "",
        "  Split assignment strategy:",
        "    Component-based (FF++ identity graph -> connected components)",
        "    Whole components assigned to train/val/test — no component ever split",
        "    Real stems mapped via identity node to same split as their fake pairings",
        "    Target proportions: train=70%, val=15%, test=15%",
        "",
        "  Identity graph statistics:",
        f"    Graph nodes              : {graph_stats.get('nodes', 0):,}",
        f"    Unique graph edges       : {graph_stats.get('unique_edges', 0):,}",
        f"    Connected components     : {graph_stats.get('components', 0):,}",
        f"    Largest component size   : {graph_stats.get('largest_component', 0):,}",
        "",
        "  Split identity counts (post-assignment):",
        f"    train : {split_identity_counts.get('train', 0):,} identities",
        f"    val   : {split_identity_counts.get('val', 0):,} identities",
        f"    test  : {split_identity_counts.get('test', 0):,} identities",
        "",
        "=" * 60,
        f"  Total accepted         : {total_acc:,}  / {sum(target_quotas.values()):,}",
        f"  Total rejected         : {total_rej:,}",
        "",
        "  Accepted counts per quota bucket:",
    ]

    for k, target in sorted(target_quotas.items()):
        got  = accepted_counts.get(k, 0)
        flag = "" if got >= target else "  *** UNDERFILLED ***"
        lines.append(f"    {k:<40}: {got:>6,} / {target:>6,}{flag}")

    lines += ["", "  Rejection reasons:"]
    for reason, count in sorted(rejection_reasons.items(), key=lambda x: -x[1]):
        lines.append(f"    {reason:<45}: {count:>6,}")

    lines += [
        "",
        "=" * 60,
        f"  Output zip size        : {zip_size:.2f} GB",
        "  S3 output destinations:",
        f"    Zip      : s3://{S3_BUCKET}/{S3_OUT_KEYS['zip']}",
        f"    Manifest : s3://{S3_BUCKET}/{S3_OUT_KEYS['manifest']}",
        f"    CSV log  : s3://{S3_BUCKET}/{S3_OUT_KEYS['csv']}",
    ]

    with open(MANIFEST_PATH, 'w') as f:
        f.write("\n".join(lines) + "\n")

    print(f"  Manifest written -> {MANIFEST_PATH}")

# MAIN

start_time = time.time()

print("\n" + "=" * 60)
print("  FF++ RetinaFace Face Extractor — SageMaker Production (Final)")
print(f"  {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")
print(f"  Target: {sum(target_quotas.values()):,} accepted faces  (10K real + 10K fake)")
print("=" * 60 + "\n")

# [1] Wipe and recreate local staging (Fix 1)
print("[1/10] Wiping and recreating local staging directories...")
clean_and_create_dirs()
make_output_split_dirs()

# [2] Download input zip from S3
print("\n[2/10] Downloading extracted frames zip from S3...")
download_from_s3(S3_INPUT_KEY, LOCAL_ZIP_IN)

if not os.path.exists(LOCAL_ZIP_IN) or os.path.getsize(LOCAL_ZIP_IN) == 0:
    raise RuntimeError(f"Input zip missing or empty after download: {LOCAL_ZIP_IN}")

# [3] Unpack via native subprocess unzip (Fix 4)
print("\n[3/10] Unpacking extracted frames (native unzip)...")
try:
    subprocess.run(
        ["unzip", "-q", LOCAL_ZIP_IN, "-d", RAW_FRAMES_DIR],
        check=True
    )
except subprocess.CalledProcessError as e:
    raise RuntimeError(f"unzip failed with return code {e.returncode}. Aborting.")

print(f"  Unpacked -> {RAW_FRAMES_DIR}")

# Free SSD after unpacking
os.remove(LOCAL_ZIP_IN)
print(f"  Input zip removed from local SSD.")

RAW_FRAMES_DIR = os.path.join(RAW_FRAMES_DIR, 'extracted_frames')

# [4] Post-unpack structure validation (Fix 3)
print("\n[4/10] Validating unpacked folder structure...")

EXPECTED_DIRS = [
    os.path.join(RAW_FRAMES_DIR, 'real'),
    os.path.join(RAW_FRAMES_DIR, 'fake'),
    os.path.join(RAW_FRAMES_DIR, 'fake', 'deepfakes'),
    os.path.join(RAW_FRAMES_DIR, 'fake', 'face2face'),
    os.path.join(RAW_FRAMES_DIR, 'fake', 'faceswap'),
    os.path.join(RAW_FRAMES_DIR, 'fake', 'neuraltextures'),
]

missing = [d for d in EXPECTED_DIRS if not os.path.isdir(d)]
if missing:
    raise RuntimeError(
        "Post-unpack structure validation failed. Missing directories:\n" +
        "\n".join(f"  {d}" for d in missing)
    )
print("  Structure validation PASSED.")

# [5] Scan frame paths per category
print("\n[5/10] Scanning frame paths per category...")

category_frames = {}

real_root  = os.path.join(RAW_FRAMES_DIR, 'real')
real_paths = glob.glob(os.path.join(real_root, '**', '*.jpg'), recursive=True)
if not real_paths:
    raise RuntimeError(f"No real frames found under {real_root}")
category_frames['real'] = real_paths
print(f"  real              : {len(real_paths):,} frames")

fake_root = os.path.join(RAW_FRAMES_DIR, 'fake')
for cat_key in FAKE_CATS:
    cat_dir = os.path.join(fake_root, cat_key)
    paths   = glob.glob(os.path.join(cat_dir, '**', '*.jpg'), recursive=True)
    if not paths:
        raise RuntimeError(f"No frames found under {cat_dir}")
    category_frames[cat_key] = paths
    print(f"  {cat_key:<20}: {len(paths):,} frames")

# [6] Build identity graph + assign splits (Fix 6)
print("\n[6/10] Building FF++ identity graph and assigning splits...")

all_fake_stems = set()
for cat_key in FAKE_CATS:
    for fp in category_frames[cat_key]:
        all_fake_stems.add(parse_source_video_stem(fp))

adjacency, all_nodes, unique_edges = build_identity_graph(all_fake_stems)
components                          = compute_connected_components(adjacency, all_nodes)

graph_stats = {
    'nodes'            : len(all_nodes),
    'unique_edges'     : len(unique_edges),
    'components'       : len(components),
    'largest_component': max((len(c) for c in components), default=0),
}

print(f"  Graph nodes              : {graph_stats['nodes']:,}")
print(f"  Unique graph edges       : {graph_stats['unique_edges']:,}")
print(f"  Connected components     : {graph_stats['components']:,}")
print(f"  Largest component size   : {graph_stats['largest_component']:,}")

node_to_split, split_identity_counts = assign_components_to_splits(components)

print(f"  Split identity allocation:")
for split in SPLIT_ORDER:
    print(f"    {split:<6}: {split_identity_counts.get(split, 0):,} identities")

# [7] Run RetinaFace extraction
print("\n[7/10] Running RetinaFace extraction engine...")
print(f"  Thresholds: confidence>={CONFIDENCE_THRESHOLD}, blur>={BLUR_THRESHOLD}, "
      f"min_face={MIN_FACE_SIZE}px, pHash Hamming<={PHASH_HAMMING_MAX}\n")

accepted_counts   = {k: 0 for k in target_quotas}
rejection_reasons = defaultdict(int)

# Per quota-bucket BK trees (Fix 8): one tree per (split x cat_suffix)
def _hash_distance(h1, h2):
    return h1 - h2

bk_trees = {k: pybktree.BKTree(_hash_distance) for k in target_quotas}

total_target = sum(target_quotas.values())
pbar = tqdm(total=total_target, desc="Securing quota", unit="face")

with open(CSV_LOG_PATH, mode='w', newline='') as log_file:
    csv_writer = csv.writer(log_file)
    csv_writer.writerow([
        "input_path", "source_video_stem", "split", "category",
        "status", "reason", "confidence", "blur_score", "phash", "bounding_box"
    ])

    for cat_key, cat_suffix in INPUT_CATEGORY_MAP.items():
        if quotas_all_filled(accepted_counts):
            break

        is_fake = (cat_key != 'real')
        print(f"  Processing: {cat_key}  ({cat_suffix})")

        process_category(
            cat_key           = cat_key,
            cat_suffix        = cat_suffix,
            frame_paths       = category_frames[cat_key],
            node_to_split     = node_to_split,
            is_fake           = is_fake,
            accepted_counts   = accepted_counts,
            rejection_reasons = rejection_reasons,
            csv_writer        = csv_writer,
            bk_trees          = bk_trees,
            pbar              = pbar,
        )

        cat_accepted = sum(accepted_counts[quota_key(s, cat_suffix)] for s in SPLIT_ORDER)
        cat_target   = sum(target_quotas[quota_key(s, cat_suffix)] for s in SPLIT_ORDER)
        print(f"  {cat_key}: {cat_accepted:,} / {cat_target:,} accepted")

pbar.close()

# [7b] Extraction summary
print("\n" + "-" * 60)
print("  EXTRACTION SUMMARY")
print("-" * 60)

total_accepted = sum(accepted_counts.values())
total_rejected = sum(rejection_reasons.values())
print(f"  Total accepted : {total_accepted:,} / {total_target:,}")
print(f"  Total rejected : {total_rejected:,}")
print("\n  Per-quota results:")

all_filled  = True
underfilled = []
for k, target in sorted(target_quotas.items()):
    got  = accepted_counts.get(k, 0)
    flag = ""
    if got < target:
        all_filled = False
        underfilled.append((k, got, target))
        flag = "  *** UNDERFILLED ***"
    print(f"    {k:<40}: {got:>6,} / {target:>6,}{flag}")

# Fix 2: Fail hard before archive/upload if any quota is underfilled
if not all_filled:
    detail = "\n".join(
        f"  {k}: {got:,} / {target:,}  (short by {target - got:,})"
        for k, got, target in underfilled
    )
    raise RuntimeError(
        f"\nQUOTA FAILURE — one or more quota buckets are underfilled.\n"
        f"Aborting before archive/upload. Local artifacts preserved for inspection.\n\n"
        f"Underfilled buckets:\n{detail}\n\n"
        f"Recommendation: increase frame buffer in the upstream extraction stage "
        f"(more frames per video), or verify all source videos were present in the input zip."
    )

print("\n  All quotas filled. Proceeding to archive and upload.")

# [8] Archive final faces
print("\n[8/10] Creating final faces archive...")
shutil.make_archive(FINAL_ZIP_BASE, 'zip', FINAL_FACES_DIR)
final_zip_path = FINAL_ZIP_BASE + '.zip'

if not os.path.exists(final_zip_path) or os.path.getsize(final_zip_path) == 0:
    raise RuntimeError(f"Final zip creation failed or empty: {final_zip_path}")

zip_size_gb = os.path.getsize(final_zip_path) / (1024 ** 3)
print(f"  Archive ready: {zip_size_gb:.2f} GB  ->  {final_zip_path}")

# [9] Write manifest + upload to S3
print("\n[9/10] Writing manifest...")
write_manifest(
    accepted_counts,
    rejection_reasons,
    final_zip_path,
    start_time,
    graph_stats,
    split_identity_counts,
)

print("\n[9b/10] Uploading outputs to S3...")
upload_targets = [
    (final_zip_path, S3_OUT_KEYS['zip']),
    (MANIFEST_PATH,  S3_OUT_KEYS['manifest']),
    (CSV_LOG_PATH,   S3_OUT_KEYS['csv']),
]

for local_path, s3_key in upload_targets:
    upload_to_s3(local_path, s3_key)

print("\n  Verifying uploads...")
for local_path, s3_key in upload_targets:
    verify_s3_upload(local_path, s3_key)
print("  All uploads verified.")

# [10] Clean local artifacts (only after verified upload)
print("\n[10/10] Cleaning local artifacts...")
shutil.rmtree(FINAL_FACES_DIR, ignore_errors=True)
shutil.rmtree(RAW_FRAMES_DIR,  ignore_errors=True)
if os.path.exists(final_zip_path):
    os.remove(final_zip_path)

print(f"  Removed: {FINAL_FACES_DIR}")
print(f"  Removed: {RAW_FRAMES_DIR}")
print(f"  Removed: {final_zip_path}")
print(f"  Logs preserved at: {LOGS_DIR}")

# ──Done──────────────────────────────────────────────────────────────────────
elapsed = time.time() - start_time
print("\n" + "=" * 60)
print("  FF++ RetinaFace Face Extractor — COMPLETE")
print(f"  Total accepted  : {total_accepted:,} faces")
print(f"  Elapsed         : {elapsed:.0f} seconds  ({elapsed/60:.1f} min)")
print(f"  Output zip      : s3://{S3_BUCKET}/{S3_OUT_KEYS['zip']}")
print(f"  Manifest        : s3://{S3_BUCKET}/{S3_OUT_KEYS['manifest']}")
print(f"  CSV log         : s3://{S3_BUCKET}/{S3_OUT_KEYS['csv']}")
print("=" * 60 + "\n")

I0000 00:00:1774710359.045124   13506 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.



  FF++ RetinaFace Face Extractor — SageMaker Production (Final)
  2026-03-28 15:06:01 UTC
  Target: 20,000 accepted faces  (10K real + 10K fake)

[1/10] Wiping and recreating local staging directories...
  Staging directories wiped and recreated under: /home/ec2-user/SageMaker/ffpp_face_stage

[2/10] Downloading extracted frames zip from S3...
  Saved → /home/ec2-user/SageMaker/ffpp_face_stage/raw_zip/ffpp_extracted_frames.zip  (5.10 GB)

[3/10] Unpacking extracted frames (native unzip)...
  Unpacked -> /home/ec2-user/SageMaker/ffpp_face_stage/raw_frames
  Input zip removed from local SSD.

[4/10] Validating unpacked folder structure...
  Structure validation PASSED.

[5/10] Scanning frame paths per category...
  real              : 20,000 frames
  deepfakes           : 5,000 frames
  face2face           : 5,000 frames
  faceswap            : 5,000 frames
  neuraltextures      : 5,000 frames

[6/10] Building FF++ identity graph and assigning splits...
  Graph nodes              : 1,00

Securing quota:   0%|          | 0/20000 [00:00<?, ?face/s]

  Processing: real  (real)


W0000 00:00:1774710419.954985   13506 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
I0000 00:00:1774710419.956264   13506 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 20833 MB memory:  -> device: 0, name: NVIDIA L4, pci bus id: 0000:31:00.0, compute capability: 8.9
I0000 00:00:1774710435.420989   13877 cuda_dnn.cc:461] Loaded cuDNN version 91002
Securing quota:  50%|█████     | 10002/20000 [20:23<11:44, 14.19face/s] 

...

Securing quota:  88%|████████▊ | 17502/20000 [34:09<03:24, 12.24face/s]

  faceswap: 2,500 / 2,500 accepted
  Processing: neuraltextures  (fake_neuraltextures)


Securing quota: 100%|█████████▉| 19961/20000 [39:32<00:04,  8.41face/s]


  neuraltextures: 2,461 / 2,500 accepted

------------------------------------------------------------
  EXTRACTION SUMMARY
------------------------------------------------------------
  Total accepted : 19,961 / 20,000
  Total rejected : 8,028

  Per-quota results:
    test_fake_deepfakes                     :    375 /    375
    test_fake_face2face                     :    375 /    375
    test_fake_faceswap                      :    375 /    375
    test_fake_neuraltextures                :    375 /    375
    test_real                               :  1,500 /  1,500
    train_fake_deepfakes                    :  1,750 /  1,750
    train_fake_face2face                    :  1,750 /  1,750
    train_fake_faceswap                     :  1,750 /  1,750
    train_fake_neuraltextures               :  1,750 /  1,750
    train_real                              :  7,000 /  7,000
    val_fake_deepfakes                      :    375 /    375
    val_fake_face2face                      :    37

RuntimeError: 
QUOTA FAILURE — one or more quota buckets are underfilled.
Aborting before archive/upload. Local artifacts preserved for inspection.

Underfilled buckets:
  val_fake_neuraltextures: 336 / 375  (short by 39)

Recommendation: increase frame buffer in the upstream extraction stage (more frames per video), or verify all source videos were present in the input zip.

Zipping and uploading to S3 manually due to Hard Failure from an Underfilled Bucket.

In [2]:
print("Bypassing quota lock and packaging 19,961 secured faces...")

# ── [8] Archive final faces ───────────────────────────────────────────────────
shutil.make_archive(FINAL_ZIP_BASE, 'zip', FINAL_FACES_DIR)
final_zip_path = FINAL_ZIP_BASE + '.zip'
zip_size_gb = os.path.getsize(final_zip_path) / (1024 ** 3)
print(f"Archive ready: {zip_size_gb:.2f} GB  ->  {final_zip_path}")

# ── [9] Write manifest + upload to S3 ────────────────────────────────────────
write_manifest(
    accepted_counts,
    rejection_reasons,
    final_zip_path,
    start_time,
    graph_stats,
    split_identity_counts,
)

print("Uploading outputs to S3...")
upload_targets = [
    (final_zip_path, S3_OUT_KEYS['zip']),
    (MANIFEST_PATH,  S3_OUT_KEYS['manifest']),
    (CSV_LOG_PATH,   S3_OUT_KEYS['csv']),
]

for local_path, s3_key in upload_targets:
    upload_to_s3(local_path, s3_key)
    verify_s3_upload(local_path, s3_key)

# ── [10] Clean local artifacts ───────────────────────────────────────────────
shutil.rmtree(FINAL_FACES_DIR, ignore_errors=True)
shutil.rmtree(RAW_FRAMES_DIR,  ignore_errors=True)
if os.path.exists(final_zip_path):
    os.remove(final_zip_path)

print("\nRescue Complete. 19,961 faces successfully secured in S3.")

Bypassing quota lock and packaging 19,961 secured faces...
Archive ready: 0.33 GB  ->  /home/ec2-user/SageMaker/ffpp_face_stage/ffpp_final_faces_20k.zip
  Manifest written -> /home/ec2-user/SageMaker/ffpp_face_stage/logs/ffpp_final_faces_20k_manifest.txt
Uploading outputs to S3...
  Uploading ffpp_final_faces_20k.zip  (0.33 GB)  → s3://deepfake-d-100k-dataset-tw26/datasets/FFPlus/final/ffpp_final_faces_20k.zip
  Verified: ffpp_final_faces_20k.zip  (0.33 GB)
  Uploading ffpp_final_faces_20k_manifest.txt  (0.00 GB)  → s3://deepfake-d-100k-dataset-tw26/datasets/FFPlus/final/ffpp_final_faces_20k_manifest.txt
  Verified: ffpp_final_faces_20k_manifest.txt  (0.00 GB)
  Uploading ffpp_final_faces_20k_log.csv  (0.01 GB)  → s3://deepfake-d-100k-dataset-tw26/datasets/FFPlus/final/ffpp_final_faces_20k_log.csv
  Verified: ffpp_final_faces_20k_log.csv  (0.01 GB)

Rescue Complete. 19,961 faces successfully secured in S3.
